# Kaggriculture: a planning agent, and how I measured it

A rule-based planning agent for Kaggriculture, with the full agent code in this notebook. It finished mid-pack, not near the top: my final pair rated about 756 (rank ~5,200 of ~10,200, leaderboard median 767). I'm sharing it for the **system design** and the **evaluation lessons**. Most of my six weeks went into one question: *did that change actually help?*

**What's here**
1. The whole agent (about 3,200 lines, standard library only), written to disk cell by cell with a short explanation of each layer.
2. A game against the built-in `starter` bot, with the replay.
3. A paired experiment that reproduces the project's biggest single gain, and shows why sample size matters.
4. A `submission.tar.gz` in the notebook output.

**Full write-up:** [paper](https://github.com/pangwangshu/competitions/blob/main/kaggriculture/report/paper.md): design, a hierarchical-RL detour, nine evaluation traps, every adopted and rejected change with confidence intervals, and an honest gap analysis. **Code and tests:** [GitHub](https://github.com/pangwangshu/competitions/tree/main/kaggriculture).

### Results in brief
- **Execution beat learning.** A min-cost-matching dispatcher was worth **+$9,968/game** on held-out seeds (t = 15.1). Six PPO submissions built on the same pipeline never beat it.
- **The interface sets the ceiling.** A *perfect* clone of the strongest public bot, replayed through my strategy interface, made only 52% of that bot's money, below my plain rule agent.
- **Same seed ≠ same game.** The engine draws a weed check for every empty tile before each shop draw. Your land use therefore changes the shops *both* players get, making paired variance about 13× larger for early-game changes.
- **Real effects, flat rating.** Eleven later changes each added $0.8k–7.2k per game with clean CIs, but my median online loss was −$17.6k, so a $3k gain overturns only about 6% of losses.

## Setup

Needs **Internet on** (Settings → Internet) to install the engine version the agent was verified against. The whole notebook runs in about five minutes on CPU.

In [ ]:
%pip install -q kaggle-environments==1.32.7
import kaggle_environments
print("kaggle-environments", kaggle_environments.__version__)

In [ ]:
import os
for d in ["farm_agent/world", "farm_agent/advisors", "farm_agent/dispatcher", "farm_agent/planners"]:
    os.makedirs(d, exist_ok=True)

## The agent

```
observation ─► world model ─► advisors ─► allocator ─► dispatcher ─► post-passes ─► action
               prices,          propose      the only      min-cost         shed room,
               forecasts,       tasks and    stage that    assignment of    sell order,
               trade flows      orders       says no       units to tasks   last-day sales
```

**Advisors propose, the allocator decides, the dispatcher moves.** Every silent-failure rule of the engine is enforced in exactly one place: the 10-order cap, the 100-item shed, and all-or-nothing `PLANT` when you oversubscribe seeds.

### Entry point and configuration
`main.py` keeps one `FarmAgent` per player index, so self-play in one process works. `config.py` holds every number, with the reasoning next to it.

In [ ]:
%%writefile main.py
"""Kaggle entry point: `agent(obs) -> action`.

The Kaggle runner exec()s this file without defining __file__ and from a
working directory that is not the agent's, so the bundled package is located
on sys.path explicitly.
"""

import os
import sys

try:
    _BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    _BASE_DIR = next(
        (d for d in [os.getcwd()] + list(sys.path)
         if d and os.path.isdir(os.path.join(d, "farm_agent"))),
        os.getcwd(),
    )
if not sys.path or sys.path[0] != _BASE_DIR:
    sys.path.insert(0, _BASE_DIR)

from farm_agent import FarmAgent  # noqa: E402

_AGENTS = {}


def agent(obs):
    """One FarmAgent per player index, recreated at the start of every episode."""
    player = obs.get("player", 0)
    if player not in _AGENTS or (obs.get("day", 0) == 0 and obs.get("hour", 0) == 0):
        _AGENTS[player] = FarmAgent()
    return _AGENTS[player].act(obs)

In [ ]:
%%writefile farm_agent/__init__.py
"""Rule-based planning agent for the Kaggle Kaggriculture competition."""

from farm_agent.agent import FarmAgent

__all__ = ["FarmAgent"]

In [ ]:
%%writefile farm_agent/config.py
"""Game constants, strategy parameters and priority bands.

The single source of truth for every number the agent uses. Advisors and the
allocator read the same priority scale, so the order in which tasks win a tile
and the exchange rate the dispatcher applies between priority and walking
distance are defined once, here.

Rationale for the less obvious values is kept next to them. Measured effects
quoted below are paired offline comparisons; see report/paper.md for method.
"""

# --------------------------------------------------------------------- game

SEASON_DAYS = 30
TURNS_PER_DAY = 24
SHED_CAPACITY = 100

CROPS = {
    "WHEAT":      {"seed": 10, "first_yield_day": 2, "max_yield_day": 4, "interval": 0, "max_yield": 6, "ongoing": False},
    "CARROT":     {"seed": 20, "first_yield_day": 2, "max_yield_day": 3, "interval": 0, "max_yield": 4, "ongoing": False},
    "TOMATO":     {"seed": 50, "first_yield_day": 8, "max_yield_day": 8, "interval": 1, "max_yield": 4, "ongoing": True},
    "STRAWBERRY": {"seed": 100, "first_yield_day": 10, "max_yield_day": 10, "interval": 2, "max_yield": 4, "ongoing": True},
    "MELON":      {"seed": 80, "first_yield_day": 10, "max_yield_day": 12, "interval": 0, "max_yield": 6, "ongoing": False},
}

ANIMALS = {
    "GOOSE": {"cost": 300, "structure": "COOP",    "first_yield_day": 4, "interval": 1, "max_held": 4, "product": "EGG"},
    "COW":   {"cost": 400, "structure": "PASTURE", "first_yield_day": 8, "interval": 2, "max_held": 6, "product": "MILK"},
    "SHEEP": {"cost": 500, "structure": "PASTURE", "first_yield_day": 6, "interval": 3, "max_held": 6, "product": "WOOL"},
}
STRUCTURE_ANIMALS = {"COOP": ("GOOSE",), "PASTURE": ("COW", "SHEEP")}

PRODUCTS = ["WHEAT", "CARROT", "TOMATO", "STRAWBERRY", "MELON", "EGG", "MILK", "WOOL", "FERTILIZER"]

SHOPS = {
    "BAKERY":         ["EGG", "WHEAT"],
    "PIZZA_SHOP":     ["MILK", "TOMATO", "WHEAT"],
    "BRUNCH_SPOT":    ["EGG", "WHEAT", "STRAWBERRY"],
    "YARN_STORE":     ["WOOL"],
    "ICE_CREAM_SHOP": ["STRAWBERRY", "MILK", "WHEAT"],
    "PET_CAFE":       ["CARROT"],
    "SMOOTHIE_SHOP":  ["STRAWBERRY", "MILK"],
    "FARMERS_MARKET": ["WHEAT", "CARROT", "TOMATO", "STRAWBERRY"],
}
SHOP_SELLS_PER_DAY = TURNS_PER_DAY // 4
SHOP_UNLOCK_INTERVAL_DAYS = 3
MAX_SHOP_INSTANCES = 8

# Price curve per product: `base` at the reference inventory MARKET_I0, rising
# below it and falling above it with a per-side shape. A port of the engine's
# own formula (tests/test_pricing.py checks it against the engine).
MARKET_I0 = 10000
PRICE_FLOOR = 1
HINGE_GAIN = 8.0
MARKET_PARAMS = {
    "WHEAT":      {"base":  25, "T": 400, "below_func": "sqrt",   "below_target": 0.80, "above_func": "log",    "above_target": 0.20},
    "CARROT":     {"base":  35, "T": 450, "below_func": "hinge",  "below_target": 1.00, "above_func": "sqrt",   "above_target": 0.70},
    "TOMATO":     {"base":  60, "T": 200, "below_func": "hinge",  "below_target": 0.40, "above_func": "sqrt",   "above_target": 0.60},
    "STRAWBERRY": {"base": 120, "T": 100, "below_func": "sqrt",   "below_target": 0.70, "above_func": "linear", "above_target": 1.60},
    "MELON":      {"base": 250, "T": 300, "below_func": "log",    "below_target": 0.20, "above_func": "sq",     "above_target": 3.60},
    "EGG":        {"base":  50, "T": 332, "below_func": "hinge",  "below_target": 0.40, "above_func": "log",    "above_target": 0.20},
    "MILK":       {"base": 160, "T": 122, "below_func": "sqrt",   "below_target": 0.60, "above_func": "linear", "above_target": 1.60},
    "WOOL":       {"base": 200, "T": 105, "below_func": "log",    "below_target": 0.20, "above_func": "sq",     "above_target": 3.20},
    "FERTILIZER": {"base": 100, "T": 200, "below_func": "linear", "below_target": 0.40, "above_func": "linear", "above_target": 0.40},
}

# The engine processes at most this many market orders per turn and silently
# drops the rest, so the allocator truncates its priority-sorted queue here.
MAX_MARKET_ORDERS = 10

# ------------------------------------------------------------ land and labour

# Prices of the second and third quadrant. The fourth ($4,000) is never bought:
# farm occupancy plateaus around 73 tiles, so its 25 tiles earned ~$354 a game
# (+$2.8k/game paired margin from dropping it; buying it *earlier* was worse).
LAND_PRICES = [1000, 2000]
LAND_EMPTY_THRESHOLD = 2  # buy the next quadrant once empty tiles fall to this

FARM_HAND_COST_MULT = 1   # engine default: the n-th hire of a day costs fib(n)

# Hiring. Hands vanish every evening and the n-th hire of a day costs fib(n),
# so hiring early in the day is free relative to hiring late. A reactive rule
# ("hire while backlog > 3x headcount") never bootstraps: on day 0 an empty
# farm has no backlog, so the season opens with one unit and locks in near
# seven. HIRE_SCHEDULE is a per-day hand-count floor under that rule (shape
# taken from the hiring curve of a public Kaggle notebook; see the paper §6.2).
HIRE_SCHEDULE = (5, 4, 4, 5, 4, 5, 8, 8, 10, 11, 11, 11, 9, 11, 10,
                 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 11, 11, 11, 11, 11)
HIRE_BACKLOG_PER_UNIT = 3  # reactive rule: chores per unit before another hire
HIRE_DAILY_CAP = 8         # cap on the reactive rule only; the schedule may exceed it
HIRE_LAST_HOUR = 12        # a hand hired later cannot do a full day's work
HIRE_CASH_FLOOR = 30       # wages are $1, 1, 2, 3, ...: hire on pocket change
HIRE_BATCH_MAX = 6         # HIRE orders per turn; leaves queue room for seeds

# Labour ledger: daily chores one unit sustains, travel included. New plantings
# are approved only while (units + hands hirable now) x this covers the load.
UNIT_DAILY_TASK_CAPACITY = 12

MONEY_RESERVE = 300        # base of the payroll floor kept before discretionary buys

# --------------------------------------------------------------- production

ROTATION_SIZE = 3          # top-N scored options served round-robin over empty tiles
SEED_BUFFER = 2

MAX_ANIMAL_STRUCTURES = 16
MAX_ANIMAL_BUYS_PER_TURN = 3
ANIMAL_BUILD_RESERVE = 600  # cash kept after building or stocking a structure
PAYBACK_MARGIN = 1.5        # an animal must return 1.5x its price over the rest of the season

# Feed. Animals eat one wheat a day and escape after two missed days.
FEED_RESERVE_PER_ANIMAL = 2
WHEAT_BUY_THRESHOLD_RATIO = 0.5  # top up when stock falls below half the reserve

# Fertilizing one-time crops: a fertilizer applied is one not sold, and cash
# early in the season is better spent on hires and animals.
FERT_VALUE_MARGIN = 1.0
FERT_CASH_FLOOR = 1000

# Days of lead time a strawberry fertilizer task is planned ahead, so that one
# application lands on the day it covers two production reads (ages 9 and 13)
# instead of one.
FERT_LEAD_DAYS = 2

# ------------------------------------------------------------------ market

# Opponent herds count at this weight when projecting supply: their standing
# animals are visible but their future upkeep is not.
OPPONENT_HERD_DISCOUNT = 0.7
SUPPLY_HORIZON_DAYS = 6

# Sell throttle for goods whose price collapses under a glut: stop selling once
# the next unit would clear below this share of base price, unless the shed is
# close to overflowing.
GLUT_SENSITIVE = {p for p, cfg in MARKET_PARAMS.items() if cfg["above_target"] > 1.0}
SELL_MIN_PRICE_RATIO = 0.3
SELL_MIN_PRICE_RATIO_OVERRIDES = {"MELON": 0.5}
SHED_HIGH_WATERMARK = 80

# Wheat is sold as a cash-flow product above the feed reserve, in batches.
WHEAT_SURPLUS_BATCH_MAX = 12
WHEAT_SELL_MIN_PRICE_RATIO = 0.6

# Realised yield per planting used when scoring a crop, where it differs from
# the model's optimal-age yield. Carrots were measured at about 2 units.
REALIZED_YIELD = {"CARROT": 2.0}

# ----------------------------------------------------------------- endgame

ENDGAME_STOP_INVEST_DAY = 24   # stop buying seeds, animals and land
ENDGAME_HARVEST_ALL_DAY = 26   # harvest anything with yield

# The episode ends after hour 22 of day 29 and the last end-of-day sweep never
# runs, so goods still carried are worth nothing. Units carrying sellable goods
# walk home in time to drop them for the final market phase.
TERMINAL_SELL_STEP = SEASON_DAYS * TURNS_PER_DAY - 2
TERMINAL_RETURN_SAFETY_TURNS = 1   # one turn of slack on the walk home
TERMINAL_RETURN_MIN_VALUE = 0.0

# ---------------------------------------------------------- priority bands

# Task priorities. Survival 900-1000, maintenance 200-920, production lowest.
PRIO_FEED_URGENT = 1000.0      # unfed yesterday: escapes tonight
PRIO_FEED = 950.0
PRIO_WATER_URGENT = 1000.0     # dry yesterday: dies tonight
PRIO_WATER = 900.0
PRIO_HARVEST = 850.0
PRIO_CARE = 800.0
PRIO_COLLECT_FERT = 750.0
# A fertilizer due today. Below harvest and watering: a bonus is only paid on a
# tile that was also watered.
PRIO_APPLY_FERT_DUE = 820.0
# A due strawberry/tomato fertilizer, which pays a doubled production that
# expires tonight. Above routine watering: the carrier arrives with the
# fertilizer and waters the same tile on its next turn, while a watering-first
# visit rarely carries fertilizer. Urgent waters (1000) still come first.
PRIO_APPLY_FERT_BERRY = 920.0
PRIO_WEED = 200.0
PRIO_PRODUCTION = 150.0
# A purchased animal waiting in the shed is idle capital.
PRIO_PLACE_ANIMAL = 960.0
PRIO_ENDGAME_HARVEST = 980.0

# Market order priorities. Queue position is settlement order, so SELLs run
# first and free the cash and shed space the buys behind them depend on.
PRIO_MKT_ENDGAME_SELL = 950.0
PRIO_MKT_SELL = 900.0
PRIO_MKT_FEED_TOPUP = 800.0
PRIO_MKT_ANIMAL = 700.0
PRIO_MKT_HIRE = 650.0
PRIO_MKT_LAND = 600.0
PRIO_MKT_SEED = 500.0

In [ ]:
%%writefile farm_agent/messages.py
"""Messages passed between pipeline stages."""

from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Set, Tuple


@dataclass
class TaskRequest:
    """A unit task proposed by an advisor: go to `pos` and perform `action`."""

    category: str                      # "feed", "water", "harvest", "place", "empty", ...
    pos: Tuple[int, int]               # target tile (x, y)
    priority: float                    # higher wins the tile; see config PRIO_*
    action: Optional[List[Any]] = None  # e.g. ["WATER"], ["PLANT", "MELON"]
    requires_inv: Dict[str, int] = field(default_factory=dict)  # items the unit must carry
    payload: Dict[str, Any] = field(default_factory=dict)


@dataclass
class MarketRequest:
    """A market order proposed by an advisor."""

    order: List[Any]                   # e.g. ["SELL", "MELON", 5], ["HIRE"]
    priority: float                    # queue position = settlement order
    category: str = "general"
    cost_estimate: float = 0.0


@dataclass
class TileClassification:
    """Own-farm tiles grouped by the work they need this turn."""

    harvest: List[Tuple[int, int]] = field(default_factory=list)
    water: List[Tuple[int, int]] = field(default_factory=list)
    weed: List[Tuple[int, int]] = field(default_factory=list)
    empty: List[Tuple[int, int]] = field(default_factory=list)
    feed: List[Tuple[int, int]] = field(default_factory=list)
    care: List[Tuple[int, int]] = field(default_factory=list)
    fert: List[Tuple[int, int]] = field(default_factory=list)        # fertilizer to collect
    apply_fert: List[Tuple[int, int]] = field(default_factory=list)  # fertilizer to apply today
    # Members of `water` whose watering today changes no output. They stay in
    # `water` (hiring and payroll read its length) but get no task.
    water_skip: Set[Tuple[int, int]] = field(default_factory=set)
    # Animal harvests deliberately left on the tile tonight (shed would overflow).
    deferred: Set[Tuple[int, int]] = field(default_factory=set)
    unoccupied_structure: List[Tuple[int, int, str]] = field(default_factory=list)  # (x, y, kind)
    animal_count: int = 0
    plant_count: int = 0

In [ ]:
%%writefile farm_agent/agent.py
"""FarmAgent: one player's decision pipeline, called once per turn.

    observation -> WorldModel -> advisors -> allocator -> dispatcher -> action

All cross-turn state (dispatcher commitments, the liquidity planner, the
market belief) lives on the instance, so two agents in one process do not
interfere.
"""

from typing import Any, Dict

from farm_agent.allocator import ResourceAllocator
from farm_agent.config import MAX_MARKET_ORDERS, SEASON_DAYS, SHED_CAPACITY
from farm_agent.dispatcher.dispatcher import Dispatcher
from farm_agent.planners.liquidity import LiquidityPlanner
from farm_agent.planners.sell_order import reorder
from farm_agent.planners.shed_room import room_orders
from farm_agent.world.state import WorldModel

PASS_ACTION = {"farmer": ["PASS"], "hands": [], "market": []}


class FarmAgent:

    def __init__(self):
        self.allocator = ResourceAllocator()
        self.dispatcher = Dispatcher()
        self.liquidity = LiquidityPlanner()

    def act(self, obs: Dict[str, Any]) -> Dict[str, Any]:
        farms = obs.get("farms", [])
        if not farms or obs.get("player", 0) >= len(farms):
            return dict(PASS_ACTION)

        world = WorldModel(obs)
        self.allocator.late_wheat.observe(obs)

        # Carriers sent home to bank cash early; may add their deposits to world.shed.
        pinned = self.liquidity.plan(world, self.allocator)
        tasks, market_orders = self.allocator.resolve(world)
        farmer_action, hand_actions = self.dispatcher.dispatch(world, tasks, pinned)
        unit_actions = [farmer_action] + hand_actions

        if world.day == SEASON_DAYS - 1:
            market_orders = self._sell_final_drops(world, unit_actions, market_orders)
        market_orders = room_orders(world, unit_actions, market_orders)
        market_orders = reorder(world, market_orders)

        action = {"farmer": farmer_action, "hands": hand_actions, "market": market_orders}
        self.allocator.late_wheat.record(action)
        return action

    @staticmethod
    def _sell_final_drops(world, unit_actions, market_orders):
        """On the last day, sell goods dropped this turn in this turn's market phase
        (unit actions settle before market orders)."""
        additions = {}
        room = max(0, SHED_CAPACITY - sum(world.shed.values()))
        for (idx, pos), action in zip(world.units, unit_actions):
            if action != ["DROP"] or not world.is_shed_adjacent(pos):
                continue
            for item, n in world.inventories[idx].items():
                take = min(max(0, n), room)
                room -= take
                if item in world.prices:
                    additions[item] = additions.get(item, 0) + take
        for item, n in additions.items():
            if not n:
                continue
            order = next((o for o in market_orders if o[:2] == ["SELL", item]), None)
            if order is not None:
                order[2] += n
            elif len(market_orders) < MAX_MARKET_ORDERS:
                market_orders.append(["SELL", item, n])
        return market_orders

### World model
- **Price model:** an exact port of the engine's price curves; a test later in this notebook checks it against the engine.
- **Dated forecasts:** crop output laid on the calendar and priced on its sale day.
- **Livestock valuation:** includes the price drop a new animal inflicts on the *opponent*, because the game is decided by the margin, not by own cash.
- **MarketBelief:** recovers the opponent's exact net trades from public inventory changes.

In [ ]:
%%writefile farm_agent/world/__init__.py
"""Perception and prediction: observation parsing, price model, forecasts."""

In [ ]:
%%writefile farm_agent/world/pricing.py
"""Market model: the engine's price curves, town demand and supply projections."""

import math
from typing import Any, Dict, List, Optional, Tuple

from farm_agent.config import (
    ANIMALS,
    CROPS,
    FEED_RESERVE_PER_ANIMAL,
    GLUT_SENSITIVE,
    HINGE_GAIN,
    MARKET_I0,
    MARKET_PARAMS,
    MAX_SHOP_INSTANCES,
    OPPONENT_HERD_DISCOUNT,
    PRICE_FLOOR,
    SEASON_DAYS,
    SELL_MIN_PRICE_RATIO,
    SELL_MIN_PRICE_RATIO_OVERRIDES,
    SHED_HIGH_WATERMARK,
    SHOP_SELLS_PER_DAY,
    SHOP_UNLOCK_INTERVAL_DAYS,
    SHOPS,
    SUPPLY_HORIZON_DAYS,
)

# ------------------------------------------------------------ price curves


def shape(func: str, x: float, T: Optional[float] = None) -> float:
    """Shape function of the price curve on one side of the reference inventory."""
    x = max(0.0, float(x))
    if func == "linear":
        return x
    if func == "sq":
        return x * x
    if func == "sqrt":
        return math.sqrt(x)
    if func == "log":
        return math.log(1.0 + x)
    if func == "hinge":
        if not T or T <= 0:
            return x
        u = x / T
        return u + HINGE_GAIN * max(0.0, u - 1.0) ** 2
    return x


def market_price(item: str, inventory: float) -> int:
    """Sale price of one unit at the given market inventory (engine formula)."""
    p = MARKET_PARAMS[item]
    base, T = p["base"], p["T"]
    if inventory < MARKET_I0:
        f = p["below_func"]
        amp = p["below_target"] * base / shape(f, T, T)
        price = base + amp * shape(f, MARKET_I0 - inventory, T)
    else:
        f = p["above_func"]
        amp = p["above_target"] * base / shape(f, T, T)
        price = base - amp * shape(f, inventory - MARKET_I0, T)
    return max(PRICE_FLOOR, round(price))


def sell_qty(item: str, qty: int, inv0: float, shed_total: int) -> int:
    """How many of `qty` units to sell now without pushing a glut-sensitive
    product below its price floor. A near-full shed overrides the throttle."""
    if qty <= 0:
        return 0
    if item not in GLUT_SENSITIVE or shed_total >= SHED_HIGH_WATERMARK:
        return qty
    ratio = SELL_MIN_PRICE_RATIO_OVERRIDES.get(item, SELL_MIN_PRICE_RATIO)
    threshold = max(PRICE_FLOOR + 1, ratio * MARKET_PARAMS[item]["base"])
    inv, n = inv0, 0
    for _ in range(qty):
        price = market_price(item, inv)
        if price < threshold:
            break
        n += 1
        if price > PRICE_FLOOR:  # a sale at the $1 floor adds no inventory
            inv += 1
    return n


def fib(n: int) -> int:
    """Wage of the (n+1)-th hire of the day: 1, 1, 2, 3, 5, 8, ..."""
    a, b = 1, 1
    for _ in range(n):
        a, b = b, a + b
    return a


# ------------------------------------------------------------- feed wheat


def feed_wheat_reserve(animal_count: int, unoccupied_structures: int, day: int) -> int:
    """Wheat to keep in the shed for feeding, shared by every buyer and seller.

    One definition matters: when the buyer (feed top-up) and the sellers used
    different reserves, the buyer bought back what the sellers had just sold,
    every turn. The reserve tapers with the days left to feed, so it is zero on
    the final day and the endgame liquidates all of it.
    """
    fed_structures = min(animal_count + unoccupied_structures, animal_count + 3)
    if fed_structures <= 0:
        return 0
    days_left = max(0, SEASON_DAYS - 1 - day)
    return min(FEED_RESERVE_PER_ANIMAL, days_left) * fed_structures


def wheat_self_sufficiency_tiles(animal_count: int, unoccupied_structures: int) -> int:
    """Growing wheat tiles needed to feed the herd from our own harvest.

    Wheat's thin per-tile margin rarely wins the crop ranking, so feed wheat is
    reserved explicitly in the rotation rather than left to compete on score.
    """
    fed_structures = min(animal_count + unoccupied_structures, animal_count + 3)
    if fed_structures <= 0:
        return 0
    wheat_yield_per_tile_per_day = CROP_PROFILE["WHEAT"]["total_yield"] / CROP_PROFILE["WHEAT"]["cycle_days"]
    return math.ceil(fed_structures / wheat_yield_per_tile_per_day)


# ------------------------------------------------------------ crop profiles


def optimal_harvest_age(crop: str) -> Tuple[int, int]:
    """(age, yield) at which daily watering stops adding yield to a one-time crop."""
    cd = CROPS[crop]
    max_yield_day, max_yield, first = cd["max_yield_day"], cd["max_yield"], cd["first_yield_day"]
    window_start = (max_yield_day + 1) // 2
    yield_units = 1
    for age in range(1, max_yield_day + 1):
        if window_start <= age <= max_yield_day:
            yield_units = min(max_yield, yield_units + 1)
        if age >= first and yield_units >= max_yield:
            return age, yield_units
    return max_yield_day, yield_units


OPTIMAL_HARVEST = {c: optimal_harvest_age(c) for c in CROPS if not CROPS[c]["ongoing"]}


def crop_profile(crop: str) -> Dict[str, Any]:
    """Cycle length, total yield and seed cost of one planting."""
    cd = CROPS[crop]
    if cd["ongoing"]:
        cycle_days = cd["first_yield_day"] + (cd["max_yield"] - 1) * cd["interval"]
        total_yield = cd["max_yield"]
    else:
        cycle_days, total_yield = OPTIMAL_HARVEST[crop]
    return {"cycle_days": max(1, cycle_days), "total_yield": total_yield, "seed_cost": cd["seed"]}


CROP_PROFILE = {c: crop_profile(c) for c in CROPS}


# ------------------------------------------------------- demand and supply


def daily_demand(product: str, unlocked_shops: List[str]) -> int:
    """Units the town consumes per day from the shops revealed so far."""
    d = 0 if product == "FERTILIZER" else 1  # town centre
    for shop in unlocked_shops:
        menu = SHOPS.get(shop, [])
        if product in menu:
            d += SHOP_SELLS_PER_DAY * (2 if len(menu) == 1 else 1)
    return d


def expected_daily_demand(product: str, unlocked_shops: List[str], day: int) -> float:
    """Current demand plus a discounted share of shops due to unlock soon."""
    slots_left = max(0, MAX_SHOP_INSTANCES - len(unlocked_shops))
    unlock_days = sum(
        1 for d in range(day + 1, day + 1 + SUPPLY_HORIZON_DAYS)
        if d % SHOP_UNLOCK_INTERVAL_DAYS == 0
    )
    expected_new = min(slots_left, unlock_days)
    per_draw = sum(
        SHOP_SELLS_PER_DAY * (2 if len(m) == 1 else 1)
        for m in SHOPS.values() if product in m
    ) / len(SHOPS)
    return daily_demand(product, unlocked_shops) + expected_new * per_draw * 0.5


def farm_weight(idx: int, own_idx: int) -> float:
    return 1.0 if idx == own_idx else OPPONENT_HERD_DISCOUNT


def product_supply_rate(farms: List[Dict[str, Any]], product: str, own_idx: int) -> float:
    """Units per day of an animal product (or fertilizer) from both standing herds."""
    rate = 0.0
    for idx, farm in enumerate(farms):
        weight = farm_weight(idx, own_idx)
        for row in farm.get("tiles", []):
            for t in row:
                if (isinstance(t, dict) and t.get("kind") in ("COOP", "PASTURE")
                        and t.get("animal")):
                    a = ANIMALS[t["animal"]]
                    if product == "FERTILIZER":
                        rate += weight
                    elif a["product"] == product:
                        rate += weight * min(a["max_held"], 1 + a["interval"]) / a["interval"]
    return rate


def projected_price(
    product: str,
    farms: List[Dict[str, Any]],
    market_inventory: Dict[str, int],
    unlocked_shops: List[str],
    day: int,
    extra_rate: float,
    own_idx: int,
) -> int:
    """Price after SUPPLY_HORIZON_DAYS of net excess supply, including `extra_rate`."""
    supply = product_supply_rate(farms, product, own_idx) + extra_rate
    demand = expected_daily_demand(product, unlocked_shops, day)
    excess = max(0.0, supply - demand)
    inv = market_inventory.get(product, MARKET_I0)
    return market_price(product, inv + excess * SUPPLY_HORIZON_DAYS)


def crop_supply_rate(farms: List[Dict[str, Any]], crop: str, own_idx: int) -> float:
    """Units per day of a crop from both farms' standing plantings."""
    per_tile_rate = CROP_PROFILE[crop]["total_yield"] / CROP_PROFILE[crop]["cycle_days"]
    rate = 0.0
    for idx, farm in enumerate(farms):
        weight = farm_weight(idx, own_idx)
        for row in farm.get("tiles", []):
            for t in row:
                if isinstance(t, dict) and t.get("kind") == "PLANT" and t.get("crop") == crop:
                    rate += weight * per_tile_rate
    return rate

In [ ]:
%%writefile farm_agent/world/forecast.py
"""Dated valuation: what one more planting is worth, sold on the day it lands.

Everything here uses the current observation only: both farms' visible tiles,
the revealed shops and the published market inventory. Strawberries and melons
are priced event by event on their sale days (their curves punish a glut and
their payoffs arrive 10+ days later); the other crops use a single projected
price at the end of their short cycle.
"""

from farm_agent.config import (
    CROPS,
    MARKET_I0,
    MAX_SHOP_INSTANCES,
    REALIZED_YIELD,
    SEASON_DAYS,
    SHOP_SELLS_PER_DAY,
    SHOP_UNLOCK_INTERVAL_DAYS,
    SHOPS,
)
from farm_agent.world.pricing import (
    CROP_PROFILE,
    crop_supply_rate,
    daily_demand,
    expected_daily_demand,
    market_price,
)

# The last day anything can still be turned into money: the season stops during
# day 29, and the end-of-day sweep that banks a backpack never runs that night.
LAST_SELL_DAY = SEASON_DAYS - 1

# Median lag, measured on both seats, from a production refresh to the sale that
# banks it: the wait to harvest, the overnight sweep into the shed, then the
# morning order.
MARKET_LAG_DAYS = 3

# Share of strawberry production days that are both watered and fertilized and
# therefore yield 2 units instead of 1 (measured ~0.75 for both players).
BONUS_RATE = 0.75
REFRESH_UNITS = 1.0 + BONUS_RATE

# Share of an animal's daily fertilizer that reaches the market.
FERT_SELL_SHARE = 0.45

DATED_CROPS = ("STRAWBERRY", "MELON")


def refresh_ages(crop):
    """Ages at which an ongoing crop produces; empty for one-time crops."""
    cd = CROPS[crop]
    if not cd["ongoing"]:
        return ()
    return tuple(cd["first_yield_day"] + i * cd["interval"] for i in range(cd["max_yield"]))


def marginal_events(crop, day):
    """(sale_day, units) a crop planted today would still bank this season."""
    events = []
    ages = refresh_ages(crop)
    if ages:
        for age in ages:
            if day + age <= LAST_SELL_DAY:
                events.append((day + age + MARKET_LAG_DAYS, REFRESH_UNITS))
    else:
        p = CROP_PROFILE[crop]
        if day + p["cycle_days"] <= LAST_SELL_DAY:
            events.append((day + p["cycle_days"] + MARKET_LAG_DAYS, float(p["total_yield"])))
    return [(min(d, LAST_SELL_DAY), n) for d, n in events]


def standing_supply(crop, day, farms):
    """{sale_day: units} that both farms' standing plantings will bring to market.

    Both farms count in full: their berries move our price exactly as much as
    ours do. Future plantings by either side are not modelled.
    """
    supply = {}

    def add(sale_day, units):
        d = min(sale_day, LAST_SELL_DAY)
        if d >= day:
            supply[d] = supply.get(d, 0.0) + units

    ages = refresh_ages(crop)
    cycle = CROP_PROFILE[crop]["cycle_days"]
    total = CROP_PROFILE[crop]["total_yield"]
    for farm in farms:
        for row in farm.get("tiles", []):
            for tile in row:
                if not (isinstance(tile, dict) and tile.get("kind") == "PLANT"
                        and tile.get("crop") == crop):
                    continue
                held = tile.get("yield_units", 0)
                if held > 0:
                    add(day + MARKET_LAG_DAYS, float(held))
                planted = tile.get("planted_day", day)
                if ages:
                    if tile.get("max_lifespan_step", -1) != -1:
                        continue  # every refresh already fired; only `held` is left
                    for age in ages:
                        if planted + age > day:
                            add(planted + age + MARKET_LAG_DAYS, REFRESH_UNITS)
                elif planted + cycle > day:
                    add(planted + cycle + MARKET_LAG_DAYS, max(0.0, total - held))
    return supply


def demand_path(crop, day, unlocked_shops):
    """{day: units} the town removes, counting unrevealed shops at the mean draw.

    One shop unlocks every SHOP_UNLOCK_INTERVAL_DAYS, drawn uniformly with
    replacement, so an unrevealed slot is worth the average menu.
    """
    base = daily_demand(crop, unlocked_shops)
    per_draw = sum(
        SHOP_SELLS_PER_DAY * (2 if len(menu) == 1 else 1)
        for menu in SHOPS.values() if crop in menu
    ) / len(SHOPS)
    slots = max(0, MAX_SHOP_INSTANCES - len(unlocked_shops))
    path, revealed = {}, 0
    for d in range(day, LAST_SELL_DAY + 1):
        path[d] = base + min(slots, revealed) * per_draw
        if d > 0 and d % SHOP_UNLOCK_INTERVAL_DAYS == 0:
            revealed += 1
    return path


def inventory_path(crop, day, farms, market_inventory, unlocked_shops):
    """Projected market inventory of `crop`, day by day to the end of the season."""
    supply = standing_supply(crop, day, farms)
    demand = demand_path(crop, day, unlocked_shops)
    inv = float(market_inventory.get(crop, MARKET_I0))
    path = {}
    for d in range(day, LAST_SELL_DAY + 1):
        inv += supply.get(d, 0.0) - demand.get(d, 0.0)
        path[d] = inv
    return path


def fertilizer_opportunity(day, farms, market_inventory):
    """What one fertilizer kept off the market would really fetch.

    No shop buys fertilizer, so every unit sold is permanent inventory and its
    price only falls. Withholding a unit forgoes the *last* unit of the season's
    stream, priced after both herds' remaining output has landed.
    """
    herd = sum(
        1
        for farm in farms
        for row in farm.get("tiles", [])
        for tile in row
        if isinstance(tile, dict) and tile.get("animal")
    )
    days_left = max(0, LAST_SELL_DAY - day)
    inv = market_inventory.get("FERTILIZER", MARKET_I0) + herd * days_left * FERT_SELL_SHARE
    return market_price("FERTILIZER", inv)


def dated_crop_score(crop, day, farms, market_inventory, unlocked_shops, own_idx):
    """Profit per occupied tile-day of one more planting, priced event by event."""
    events = marginal_events(crop, day)
    if not events:
        return float("-inf")
    path = inventory_path(crop, day, farms, market_inventory, unlocked_shops)
    revenue, extra = 0.0, 0.0
    for sale_day, units in events:
        d = min(max(sale_day, day), LAST_SELL_DAY)
        # Price the lot at its own midpoint so it pays for the impact it makes.
        revenue += units * market_price(crop, path.get(d, float(MARKET_I0)) + extra + units / 2.0)
        extra += units
    cost = float(CROPS[crop]["seed"])
    ages = refresh_ages(crop)
    if ages:
        # One fertilizer covers two production reads, charged at the rate the
        # bonus is actually achieved.
        cost += BONUS_RATE * ((len(events) + 1) // 2) * fertilizer_opportunity(
            day, farms, market_inventory
        )
    occupied = min(CROP_PROFILE[crop]["cycle_days"], LAST_SELL_DAY - day)
    return (revenue - cost) / max(1, occupied)


def crop_score(crop, day, farms, market_inventory, unlocked_shops, own_idx):
    """Profit per tile-day of planting `crop` today; -inf if it cannot pay back."""
    if crop in DATED_CROPS:
        return dated_crop_score(crop, day, farms, market_inventory, unlocked_shops, own_idx)
    p = CROP_PROFILE[crop]
    if day + p["cycle_days"] > SEASON_DAYS:
        return float("-inf")
    supply = crop_supply_rate(farms, crop, own_idx)
    demand = expected_daily_demand(crop, unlocked_shops, day)
    inv = market_inventory.get(crop, MARKET_I0)
    price = market_price(crop, inv + (supply - demand) * p["cycle_days"])
    revenue = REALIZED_YIELD.get(crop, p["total_yield"]) * price
    return (revenue - p["seed_cost"]) / p["cycle_days"]


def fert_coverage(crop, planted_day, day, fertilized_until_day, apply_day):
    """Production reads a fertilizer applied on `apply_day` would newly pay a bonus on.

    An application covers the reads on apply_day..apply_day+2 (the engine sets
    fertilized_until_day = apply_day + 2). Reads already covered, or whose
    output would land after LAST_SELL_DAY, do not count.
    """
    covered = 0
    for age in refresh_ages(crop):
        produce_day = planted_day + age
        read_day = produce_day - 1
        if read_day < apply_day or read_day > apply_day + 2:
            continue
        if produce_day <= day or produce_day > LAST_SELL_DAY:
            continue
        if fertilized_until_day >= read_day:
            continue
        covered += 1
    return covered

In [ ]:
%%writefile farm_agent/world/livestock.py
"""Calendar valuation of a marginal animal, and the purchase plan built on it.

A marginal animal is valued by laying out its production days (care bonuses
included) on the calendar and pricing each lot against the projected market on
its sale day. Two price effects are counted:

- own displacement: our extra output lowers the price our existing herd gets;
- rival displacement, at half weight: it also lowers the price the opponent
  gets. The game is decided by the margin, not by our own cash, so a sale that
  depresses a contested market is worth more than the cash it brings in.
  Counting it shifted the herd from geese toward cows and sheep and was worth
  about +$2.9k/game on a held-out panel (report/paper.md §8.2).
"""

from farm_agent.config import (
    ANIMAL_BUILD_RESERVE,
    ANIMALS,
    MARKET_I0,
    MAX_ANIMAL_BUYS_PER_TURN,
    PAYBACK_MARGIN,
    PRIO_MKT_ANIMAL,
    SEASON_DAYS,
)
from farm_agent.messages import MarketRequest
from farm_agent.world.forecast import demand_path
from farm_agent.world.pricing import market_price, projected_price

CARE_RATE = 0.95        # share of days an animal is actually cared for
SALE_LAG = 1            # days from production to sale
PLACEMENT_LAG = 1       # a purchase is placed the next day at the earliest
RIVAL_PRICE_IMPACT_WEIGHT = 0.5


def events(animal, placed, day, pending=0, held=0):
    """{sale_day: units} an animal will still deliver this season.

    The care bonus banked so far is paid with the next production; production
    whose sale would fall after the season is dropped.
    """
    a = ANIMALS[animal]
    result = {}
    if held and day + SALE_LAG < SEASON_DAYS:
        result[day + SALE_LAG] = float(held)
    bank = float(pending)
    for current in range(day, SEASON_DAYS - 1):
        if current < placed:
            continue
        age = current + 1 - placed
        if age >= a["first_yield_day"] and (age - a["first_yield_day"]) % a["interval"] == 0:
            sale = current + 1 + SALE_LAG
            if sale < SEASON_DAYS:
                result[sale] = result.get(sale, 0.0) + min(a["max_held"], 1.0 + bank)
            bank = 0.0
        bank += CARE_RATE
    return result


def dated_animal_score(animal, day, farms, market_inventory, unlocked_shops, prices, own_idx):
    """Net value per remaining day of buying one more `animal` today; -inf if it cannot pay back."""
    a = ANIMALS[animal]
    marginal = events(animal, day + PLACEMENT_LAG, day)
    if not marginal:
        return float("-inf")
    product = a["product"]
    supply, own, rival = {}, {}, {}
    for seat, farm in enumerate(farms):
        for row in farm.get("tiles", []):
            for tile in row:
                if not isinstance(tile, dict) or tile.get("animal") != animal:
                    continue
                for d, n in events(animal, tile["placed_day"], day,
                                   tile.get("pending_care_bonus", 0), tile.get("yield_units", 0)).items():
                    supply[d] = supply.get(d, 0.0) + n
                    if seat == own_idx:
                        own[d] = own.get(d, 0.0) + n
                    else:
                        rival[d] = rival.get(d, 0.0) + n
    demand = demand_path(product, day, unlocked_shops)
    inventory = float(market_inventory.get(product, MARKET_I0))
    extra = revenue = displacement = rival_loss = 0.0
    for d in range(day, SEASON_DAYS):
        inventory -= demand[d]
        base_units = supply.get(d, 0.0)
        n = marginal.get(d, 0.0)
        # Quote mid-lot so each lot pays for its own price impact.
        quote_inv = inventory + base_units / 2
        price_before = market_price(product, quote_inv)
        price_after = market_price(product, quote_inv + extra + n / 2)
        revenue += n * price_after
        displacement += own.get(d, 0.0) * (price_before - price_after)
        rival_loss += rival.get(d, 0.0) * (price_before - price_after)
        inventory += base_units
        extra += n
    remaining = SEASON_DAYS - day
    fert_price = projected_price("FERTILIZER", farms, market_inventory,
                                 unlocked_shops, day, 1.0, own_idx)
    net = revenue + (fert_price - prices.get("WHEAT", 0)) * remaining
    net -= displacement
    net += RIVAL_PRICE_IMPACT_WEIGHT * rival_loss
    if net < a["cost"] * PAYBACK_MARGIN:
        return float("-inf")
    return (net - a["cost"]) / remaining


# ------------------------------------------------------------ purchase plan


def planning_farms(world):
    """Both farms, with animals we own but have not placed yet added to ours.

    The additions exist only for valuation; the observation is not modified.
    """
    farms = list(world.farms)
    own = dict(farms[world.player_idx])
    own["tiles"] = list(own["tiles"]) + [[]]
    farms[world.player_idx] = own
    for animal in ANIMALS:
        stock = world.shed.get(animal, 0) + sum(i.get(animal, 0) for i in world.inventories)
        for _ in range(stock):
            add_animal(farms, world, animal)
    return farms


def add_animal(farms, world, animal):
    farms[world.player_idx]["tiles"][-1].append(dict(
        kind=ANIMALS[animal]["structure"], animal=animal,
        placed_day=world.day + 1, yield_units=0, pending_care_bonus=0))


def score(world, farms, animal):
    return dated_animal_score(animal, world.day, farms, world.market_inv,
                              world.unlocked_shops, world.prices, world.player_idx)


def buy_requests(world):
    """BUY_ANIMAL orders for free structures, re-valuing after each purchase."""
    farms = planning_farms(world)
    free = {kind: sum(t[2] == kind for t in world.classified_tiles.unoccupied_structure)
            for kind in ("PASTURE", "COOP")}
    for a, data in ANIMALS.items():
        free[data["structure"]] -= world.shed.get(a, 0) + sum(i.get(a, 0) for i in world.inventories)
    budget, counts, requests = world.money, {}, []
    for _ in range(MAX_ANIMAL_BUYS_PER_TURN):
        eligible = [a for a, d in ANIMALS.items() if free[d["structure"]] > 0
                    and budget - d["cost"] >= ANIMAL_BUILD_RESERVE]
        if not eligible:
            break
        values = [(score(world, farms, a), a) for a in eligible]
        value, animal = max(values)
        if value == float("-inf"):
            break
        budget -= ANIMALS[animal]["cost"]
        free[ANIMALS[animal]["structure"]] -= 1
        counts[animal] = counts.get(animal, 0) + 1
        add_animal(farms, world, animal)
    for animal, n in counts.items():
        requests.append(MarketRequest(order=["BUY_ANIMAL", animal, n], priority=PRIO_MKT_ANIMAL,
                                      category="animal_stock", cost_estimate=n * ANIMALS[animal]["cost"]))
    return requests

In [ ]:
%%writefile farm_agent/world/market_belief.py
"""MarketBelief: exact market trade flows recovered from public observations.

The engine settles each turn as unit actions, then market orders, then town
consumption, so for every product

    inventory[t+1] - inventory[t] = sum over both players (units sold above $1 - units bought)
                                    - town consumption at t

Our own trades are known exactly (our orders plus our shed after unit
actions), and town consumption follows from the revealed shops, so the
opponent's net flow is recovered exactly, up to its sales at the $1 floor,
which leave no trace in inventory. The identity held with zero error over
1.9M product-turns of recorded games.

The flow history feeds a short-horizon inventory forecast for wheat and
fertilizer, the two goods both players also buy back from the market.
"""

from farm_agent.config import (
    MARKET_I0,
    MAX_SHOP_INSTANCES,
    PRODUCTS,
    SEASON_DAYS,
    SHED_CAPACITY,
    SHOP_SELLS_PER_DAY,
    SHOP_UNLOCK_INTERVAL_DAYS,
    SHOPS,
    TURNS_PER_DAY,
)
from farm_agent.world.pricing import market_price

ANIMALS = ("GOOSE", "COW", "SHEEP")
SHED_ACCESS = ((4, 4), (5, 4), (4, 5), (5, 5))
LAST_DAY = SEASON_DAYS - 1

# ------------------------------------------------------------- accounting


def town_consumption(step, shops):
    """{item: units} the town removes after `step`'s market phase."""
    out = {}
    if step % 4 == 0:
        for name in shops:
            menu = SHOPS.get(name, ())
            for item in menu:
                out[item] = out.get(item, 0) + (2 if len(menu) == 1 else 1)
    if step % TURNS_PER_DAY == 0:
        for item in PRODUCTS:
            if item != "FERTILIZER":
                out[item] = out.get(item, 0) + 1
    return out


def _unit_positions(farm):
    return [tuple(farm["farmer"])] + [tuple(h) for h in farm.get("hands", [])]


def shed_after_units(obs, act):
    """(shed, carried) after this turn's unit actions.

    Only PICKUP, PLACE and DROP touch the shed, and units act in farmer-then-
    hands order, so the running shed is exact without simulating tiles.
    """
    me = obs["player"]
    farm = obs["farms"][me]
    shed = {k: v for k, v in obs["private"]["shed"].items()}
    invs = [dict(i) for i in obs["private"]["inventories"]]
    positions = _unit_positions(farm)
    actions = [act.get("farmer") or ["PASS"]] + list(act.get("hands") or [])
    tiles = farm["tiles"]
    for idx, pos in enumerate(positions):
        a = actions[idx] if idx < len(actions) else ["PASS"]
        if not isinstance(a, list) or not a:
            continue
        while len(invs) <= idx:
            invs.append({})
        inv = invs[idx]
        op = a[0]
        adjacent = pos in SHED_ACCESS
        if op == "DROP" and adjacent:
            for item, n in list(inv.items()):
                if n > 0:
                    room = max(0, SHED_CAPACITY - sum(shed.values()))
                    take = min(n, room)
                    if take:
                        shed[item] = shed.get(item, 0) + take
                del inv[item]
        elif op == "PICKUP" and adjacent and len(a) >= 2:
            try:
                n = int(a[2]) if len(a) >= 3 else 1
            except (TypeError, ValueError):
                continue
            n = min(n, shed.get(a[1], 0))
            if n > 0:
                shed[a[1]] -= n
                inv[a[1]] = inv.get(a[1], 0) + n
        elif op == "PLACE" and len(a) >= 2:
            item = a[1]
            tile = tiles[pos[1]][pos[0]]
            if (item in ANIMALS and isinstance(tile, dict) and tile.get("kind") in ("COOP", "PASTURE")
                    and "animal" not in tile):
                continue  # placing an animal on its structure, not into the shed
            if not adjacent:
                continue
            try:
                n = int(a[2]) if len(a) >= 3 else 1
            except (TypeError, ValueError):
                continue
            n = min(n, inv.get(item, 0), max(0, SHED_CAPACITY - sum(shed.values())))
            if n > 0:
                inv[item] -= n
                if inv[item] == 0:
                    del inv[item]
                shed[item] = shed.get(item, 0) + n
    return shed, invs


def own_trades(obs, act, nxt):
    """{item: (sold, bought)} our market phase committed on this transition.

    A SELL commits min(ordered, shed) in queue order. Buys are read from shed
    conservation, except on the last turn of a day, when the overnight sweep
    hides them and they are taken as ordered, capped by shed room.
    """
    shed, _ = shed_after_units(obs, act)
    running = dict(shed)
    sold, bought = {}, {}
    eod = obs["hour"] == TURNS_PER_DAY - 1
    for order in (act.get("market") or [])[:10]:
        if not isinstance(order, list) or len(order) < 3 or order[0] not in ("SELL", "BUY_PRODUCT", "BUY_ANIMAL"):
            continue
        try:
            n = int(order[2])
        except (TypeError, ValueError):
            continue
        item = order[1]
        if n <= 0:
            continue
        if order[0] == "SELL":
            k = min(n, running.get(item, 0))
            running[item] = running.get(item, 0) - k
            sold[item] = sold.get(item, 0) + k
        elif eod:
            room = max(0, SHED_CAPACITY - sum(running.values()))
            k = min(n, room)
            running[item] = running.get(item, 0) + k
            if order[0] == "BUY_PRODUCT":
                bought[item] = bought.get(item, 0) + k
    if not eod and nxt is not None:
        after = nxt["private"]["shed"]
        for item in ("WHEAT", "FERTILIZER"):
            k = after.get(item, 0) - (shed.get(item, 0) - sold.get(item, 0))
            if k > 0:
                bought[item] = k
    return {i: (sold.get(i, 0), bought.get(i, 0)) for i in set(sold) | set(bought)}


def opponent_net_flow(obs, own, nxt):
    """{item: opponent units sold above the floor minus units bought} on this transition."""
    before = obs["market"]["inventory"]
    after = nxt["market"]["inventory"]
    town = town_consumption(obs["step"], obs["town"]["unlocked_shops"])
    out = {}
    for item in PRODUCTS:
        s, b = own.get(item, (0, 0))
        inv, above = before[item] - b, 0
        for _ in range(s):
            if market_price(item, inv) > 1:
                inv += 1
                above += 1
        out[item] = after[item] - before[item] + town.get(item, 0) - (above - b)
    return out


def _copy_tiles(tiles):
    return [[dict(t) if isinstance(t, dict) else t for t in row] for row in tiles]


def snapshot(obs):
    """Plain copies of every observation field the belief reads."""
    farms = []
    for f in obs["farms"]:
        farms.append(dict(tiles=_copy_tiles(f["tiles"]), farmer=list(f["farmer"]),
                          hands=[list(h) for h in f.get("hands", [])], money=f.get("money", 0),
                          hires_today=f.get("hires_today", 0)))
    private = obs["private"]
    return dict(step=obs["day"] * TURNS_PER_DAY + obs["hour"], day=obs["day"], hour=obs["hour"],
                player=obs["player"], farms=farms,
                private=dict(shed=dict(private["shed"]), inventories=[dict(i) for i in private["inventories"]],
                             seeds=dict(private.get("seeds", {}))),
                market=dict(inventory=dict(obs["market"]["inventory"]), prices=dict(obs["market"]["prices"])),
                town=dict(unlocked_shops=list(obs["town"]["unlocked_shops"])))


class MarketBelief:
    """Per-day net flows (sold above floor minus bought) for each player.

    Call update(obs, last_action) at the start of every turn, where last_action
    is the action this agent returned for the previous observation.
    """

    def __init__(self):
        self.prev = None
        self.flow = {"own": {}, "opp": {}}  # role -> day -> item -> net units

    @staticmethod
    def _bump(table, day, delta):
        row = table.setdefault(day, {})
        for item, n in delta.items():
            if n:
                row[item] = row.get(item, 0) + n

    def update(self, obs, last_action):
        cur = snapshot(obs)
        prev = self.prev
        if prev is not None and last_action is not None and cur["step"] == prev["step"] + 1:
            own = own_trades(prev, last_action, cur)
            flow = opponent_net_flow(prev, own, cur)
            day = prev["day"]
            inv0 = prev["market"]["inventory"]
            own_net = {}
            for item, (s, b) in own.items():
                inv, above = inv0[item] - b, 0
                for _ in range(s):
                    if market_price(item, inv) > 1:
                        inv += 1
                        above += 1
                own_net[item] = above - b
            self._bump(self.flow["own"], day, own_net)
            self._bump(self.flow["opp"], day, flow)
        self.prev = cur
        return cur


# --------------------------------------------------------------- forecast

_FLOOR_INV = {}


def floor_inventory(item):
    """Smallest inventory at which the quote reaches $1 (sales there add nothing)."""
    if item not in _FLOOR_INV:
        lo, hi = MARKET_I0, MARKET_I0 + 200000
        while lo < hi:
            mid = (lo + hi) // 2
            if market_price(item, mid) <= 1:
                hi = mid
            else:
                lo = mid + 1
        _FLOOR_INV[item] = lo
    return _FLOOR_INV[item]


def demand_by_day(item, day, shops, last):
    """{D: units the town removes on day D}, unrevealed shop slots at the mean draw."""
    base_tick = sum((2 if len(SHOPS[s]) == 1 else 1) for s in shops if item in SHOPS.get(s, ()))
    per_draw = sum((2 if len(m) == 1 else 1) for m in SHOPS.values() if item in m) / len(SHOPS)
    out = {}
    extra = 0
    for D in range(day, last + 1):
        if D > day and D % SHOP_UNLOCK_INTERVAL_DAYS == 0 and len(shops) + extra < MAX_SHOP_INSTANCES:
            extra += 1
        out[D] = SHOP_SELLS_PER_DAY * (base_tick + extra * per_draw) + (0 if item == "FERTILIZER" else 1)
    return out


def roll(inv0, item, day, horizon, supply, demand):
    """Inventory at day+horizon from per-day supply and demand, honouring the $1 floor."""
    inv = float(inv0)
    cap = floor_inventory(item)
    for D in range(day, day + horizon):
        before = inv
        inv += supply.get(D, 0.0)
        if inv > cap:
            inv = max(before, float(cap))
        inv -= demand.get(D, 0.0)
    return inv


def recent_rate(belief, role, item, day, window):
    days = [D for D in range(day - window, day) if D >= 0]
    if not days:
        return 0.0
    return sum(belief.flow[role].get(D, {}).get(item, 0) for D in days) / len(days)


def flow_inventory(belief, obs, item, horizon, window=3):
    """Market inventory in `horizon` days if both players keep their recent net flow."""
    day = obs["day"]
    shops = obs["town"]["unlocked_shops"]
    rate = recent_rate(belief, "own", item, day, window) + recent_rate(belief, "opp", item, day, window)
    supply = {D: rate for D in range(day, day + horizon)}
    demand = demand_by_day(item, day, shops, min(LAST_DAY, day + horizon))
    return roll(obs["market"]["inventory"][item], item, day, horizon, supply, demand)

In [ ]:
%%writefile farm_agent/world/state.py
"""WorldModel: the parsed observation plus a per-turn classification of own tiles."""

from typing import Any, Dict, List, Optional, Tuple

from farm_agent.config import CROPS, FERT_CASH_FLOOR, FERT_LEAD_DAYS, FERT_VALUE_MARGIN
from farm_agent.messages import TileClassification
from farm_agent.planners.shed_room import deferred_harvests
from farm_agent.world import forecast
from farm_agent.world.pricing import OPTIMAL_HARVEST, market_price


def water_pays(cd, tile, day):
    """Does watering this plant on `day` change any output?

    Ongoing crop: tonight's refresh reads a production. One-time crop: its age
    is in the bonus window, where each watering adds yield.
    """
    age = day - tile.get("planted_day", day)
    if cd["ongoing"]:
        k = age + 1 - cd["first_yield_day"]
        return k >= 0 and k % cd["interval"] == 0 and k // cd["interval"] < cd["max_yield"]
    return (cd["max_yield_day"] + 1) // 2 <= age <= cd["max_yield_day"]


def rule_skips(cd, tile, day):
    """A plant watered at the last refresh survives one dry day, so its watering
    is skipped on days when watering pays nothing. Freed labour went to harvests
    and fertilizing: same output, far fewer waterings."""
    return not water_pays(cd, tile, day)


def planned_skip(cd, tile, day):
    """True if a plant is one dry day behind only because yesterday's watering
    was deliberately skipped (as opposed to a genuine miss)."""
    return (tile.get("consecutive_unwatered", 0) == 1
            and tile.get("planted_day", day) <= day - 2
            and rule_skips(cd, tile, day - 1))


class WorldModel:
    """The current observation, with fast queries and a cached tile classification."""

    def __init__(self, obs: Dict[str, Any]):
        self.obs = obs
        self.player_idx = obs.get("player", 0)
        self.step = obs.get("step", 0)
        self.day = obs.get("day", 0)
        self.hour = obs.get("hour", 0)

        self.farms = obs.get("farms", [])
        if self.farms and self.player_idx < len(self.farms):
            self.my_farm = self.farms[self.player_idx]
        else:
            self.my_farm = {"money": 0, "tiles": [], "farmer": [0, 0], "hands": []}

        self.money: int = self.my_farm.get("money", 0)
        self.tiles: List[List[Any]] = self.my_farm.get("tiles", [])
        self.board_size: int = len(self.tiles)

        self.private: Dict[str, Any] = obs.get("private") or {}
        self.shed: Dict[str, int] = self.private.get("shed") or {}
        self.seeds: Dict[str, int] = self.private.get("seeds") or {}
        self.inventories: List[Dict[str, int]] = self.private.get("inventories") or [{}]

        self.market: Dict[str, Any] = obs.get("market") or {}
        self.prices: Dict[str, int] = self.market.get("prices") or {}
        self.market_inv: Dict[str, int] = self.market.get("inventory") or {}

        self.town: Dict[str, Any] = obs.get("town") or {}
        self.unlocked_shops: List[str] = self.town.get("unlocked_shops") or []

        # Farmer is unit 0; hands are 1, 2, ...
        self.units: List[Tuple[int, Tuple[int, int]]] = [
            (0, tuple(self.my_farm.get("farmer", [0, 0])))
        ] + [
            (i + 1, tuple(pos))
            for i, pos in enumerate(self.my_farm.get("hands", []))
        ]

        self._classified_tiles: Optional[TileClassification] = None

    @property
    def classified_tiles(self) -> TileClassification:
        if self._classified_tiles is None:
            self._classified_tiles = self._classify()
        return self._classified_tiles

    def _classify(self) -> TileClassification:
        tc = TileClassification()
        for y in range(self.board_size):
            for x in range(self.board_size):
                tile = self.tiles[y][x]
                if tile is None:
                    tc.empty.append((x, y))
                    continue
                if tile == "LOCKED" or not isinstance(tile, dict):
                    continue

                kind = tile.get("kind")
                if kind == "WEED":
                    tc.weed.append((x, y))
                elif kind == "PLANT":
                    self._classify_plant(tc, x, y, tile)
                elif kind in ("COOP", "PASTURE"):
                    if tile.get("animal") is None:
                        tc.unoccupied_structure.append((x, y, kind))
                    else:
                        tc.animal_count += 1
                        if tile.get("yield_units", 0) > 0:
                            tc.harvest.append((x, y))
                        if not tile.get("fed_today"):
                            tc.feed.append((x, y))
                        if not tile.get("cared_today"):
                            tc.care.append((x, y))
                        if tile.get("fertilizer_available"):
                            tc.fert.append((x, y))
        # A fertilizer task no unit can deliver before the day ends would only
        # hold stock off the market and take a dispatcher slot.
        for pos in list(tc.apply_fert):
            if not self._fert_reachable(pos):
                tc.apply_fert.remove(pos)
        # On a night the shed would overflow, animal products wait on the tile.
        tc.deferred = deferred_harvests(self, tc)
        if tc.deferred:
            tc.harvest = [p for p in tc.harvest if p not in tc.deferred]
        return tc

    def _classify_plant(self, tc: TileClassification, x: int, y: int, tile: Dict[str, Any]) -> None:
        crop = tile.get("crop")
        if not crop or crop not in CROPS:
            return
        tc.plant_count += 1
        cd = CROPS[crop]
        age = self.day - tile.get("planted_day", self.day)
        yield_units = tile.get("yield_units", 0)
        can_harvest = False
        if yield_units > 0 and age >= cd["first_yield_day"]:
            # A fertilized one-time crop can hit max yield before its usual
            # optimal age; harvest as soon as it is capped.
            can_harvest = (
                True
                if cd["ongoing"]
                else age >= OPTIMAL_HARVEST[crop][0] or yield_units >= cd["max_yield"]
            )
        spent = cd["ongoing"] and tile.get("max_lifespan_step", -1) != -1

        if can_harvest:
            tc.harvest.append((x, y))
        elif spent:
            pass  # an ongoing crop past its last production, decaying to a weed
        elif not tile.get("watered_today"):
            tc.water.append((x, y))
            if tile.get("consecutive_unwatered", 0) == 0 and rule_skips(cd, tile, self.day):
                tc.water_skip.add((x, y))

        if can_harvest or spent or tile.get("fertilized_until_day", -1) >= self.day:
            return
        if cd["ongoing"]:
            due = self._fert_service_day(tile, crop)
            if due is not None and due <= self.day:
                tc.apply_fert.append((x, y))
        elif self.money >= FERT_CASH_FLOOR:
            # One-time crop: each watering in the bonus window yields +2 instead
            # of +1 while fertilized. A fertilizer applied is one not sold, so
            # require the extra units to out-value it.
            window_start = (cd["max_yield_day"] + 1) // 2
            if window_start <= age <= cd["max_yield_day"] and yield_units < cd["max_yield"]:
                extra = min(
                    3,  # a fertilizer lasts 3 days
                    cd["max_yield_day"] - age + 1,
                    cd["max_yield"] - yield_units,
                )
                crop_price = self.prices.get(crop, 0)
                fert_price = self.prices.get("FERTILIZER", 0)
                if extra * crop_price > FERT_VALUE_MARGIN * fert_price:
                    tc.apply_fert.append((x, y))

    def _fert_service_day(self, tile, crop):
        """The day to bring a fertilizer to this ongoing crop, or None to skip it.

        Picks, within FERT_LEAD_DAYS, the application day that covers the most
        production reads, then checks that the tile will be wet when they are
        read, and that the covered berries at their projected sale price beat
        what the fertilizer itself would fetch.
        """
        planted = tile.get("planted_day", self.day)
        fu = tile.get("fertilized_until_day", -1)
        best_day, best_cov = None, 0
        for ahead in range(0, FERT_LEAD_DAYS + 1):
            cov = forecast.fert_coverage(crop, planted, self.day, fu, self.day + ahead)
            if cov > best_cov:
                best_day, best_cov = self.day + ahead, cov
        if best_day is None:
            return None
        # A dry read pays no bonus, and a plant two dry days from death is not
        # worth stocking.
        if tile.get("consecutive_unwatered", 0) >= 2:
            return None
        sale_day = min(best_day + 1 + forecast.MARKET_LAG_DAYS, forecast.LAST_SELL_DAY)
        path = forecast.inventory_path(crop, self.day, self.farms, self.market_inv,
                                       self.unlocked_shops)
        berry = market_price(crop, path.get(sale_day, forecast.MARKET_I0))
        if best_cov * berry <= forecast.fertilizer_opportunity(
                self.day, self.farms, self.market_inv):
            return None
        return best_day

    def _fert_reachable(self, pos):
        """True if some unit could still fetch a fertilizer and reach `pos` today."""
        if not self.units:
            return False
        carried = max((inv.get("FERTILIZER", 0) for inv in self.inventories), default=0)
        if carried <= 0 and self.shed.get("FERTILIZER", 0) <= 0:
            return False
        turns_left = max(0, self.obs.get("turns_per_day", 24) - self.hour)
        for idx, unit in self.units:
            direct = abs(unit[0] - pos[0]) + abs(unit[1] - pos[1])
            if idx < len(self.inventories) and self.inventories[idx].get("FERTILIZER", 0) > 0:
                if direct <= turns_left:
                    return True
                continue
            if self.shed.get("FERTILIZER", 0) <= 0:
                continue
            shed_tile = self.nearest_shed_tile(unit[0], unit[1])
            trip = (abs(unit[0] - shed_tile[0]) + abs(unit[1] - shed_tile[1])
                    + abs(shed_tile[0] - pos[0]) + abs(shed_tile[1] - pos[1]))
            if trip + 1 <= turns_left:
                return True
        return False

    def shed_access_tiles(self) -> List[Tuple[int, int]]:
        """The 4 centre tiles from which a unit can use the shed."""
        half = self.board_size // 2
        return [(half - 1, half - 1), (half, half - 1), (half - 1, half), (half, half)]

    def is_shed_adjacent(self, pos: Tuple[int, int]) -> bool:
        return tuple(pos) in self.shed_access_tiles()

    def nearest_shed_tile(self, fx: int, fy: int) -> Tuple[int, int]:
        return min(
            self.shed_access_tiles(),
            key=lambda t: abs(t[0] - fx) + abs(t[1] - fy),
        )

### Advisors
Small modules that each look after one concern and emit prioritised `TaskRequest`s and `MarketRequest`s. Priorities share one scale: survival 900–1000, maintenance 200–920, planting 150.

In [ ]:
%%writefile farm_agent/advisors/__init__.py
"""Advisors: each proposes tasks and market orders from one concern."""

In [ ]:
%%writefile farm_agent/advisors/base.py
"""Advisor interface."""

from typing import List, Tuple

from farm_agent.messages import MarketRequest, TaskRequest
from farm_agent.world.state import WorldModel


class BaseAdvisor:
    """Advisors read the world and propose work; they never act or veto."""

    def get_requests(self, world: WorldModel) -> Tuple[List[TaskRequest], List[MarketRequest]]:
        return [], []

In [ ]:
%%writefile farm_agent/advisors/survival.py
"""Survival: water plants and feed animals before they are lost."""

from typing import List, Tuple

from farm_agent.advisors.base import BaseAdvisor
from farm_agent.config import CROPS, PRIO_FEED, PRIO_FEED_URGENT, PRIO_WATER, PRIO_WATER_URGENT, TURNS_PER_DAY
from farm_agent.dispatcher.pathfinding import manhattan_dist
from farm_agent.messages import MarketRequest, TaskRequest
from farm_agent.planners.late_wheat import is_late_wheat
from farm_agent.world.state import WorldModel, planned_skip

# A deliberately skipped watering escalates to urgent only once the nearest
# unit has fewer than this many turns of slack left today.
WATER_DEFER_SLACK = 6
PRIO_LATE_WHEAT_WATER = 890.0
PRIO_LATE_WHEAT_WATER_URGENT = 940.0


class SurvivalAdvisor(BaseAdvisor):
    """Two missed days kill a plant or lose an animal; the second miss is urgent."""

    def get_requests(self, world: WorldModel) -> Tuple[List[TaskRequest], List[MarketRequest]]:
        tasks: List[TaskRequest] = []
        tc = world.classified_tiles

        for pos in tc.feed:
            x, y = pos
            tile = world.tiles[y][x]
            unfed = tile.get("consecutive_unfed", 0) if isinstance(tile, dict) else 0
            prio = PRIO_FEED_URGENT if unfed >= 1 else PRIO_FEED
            tasks.append(TaskRequest(category="feed", pos=pos, priority=prio,
                                     action=["FEED"], requires_inv={"WHEAT": 1}))

        for pos in tc.water:
            if pos in tc.water_skip:
                continue  # watering today changes no output
            x, y = pos
            tile = world.tiles[y][x]
            # A new plant starts one dry day behind, so its first day is urgent.
            unwatered = tile.get("consecutive_unwatered", 0) if isinstance(tile, dict) else 0
            prio = PRIO_WATER_URGENT if unwatered >= 1 else PRIO_WATER
            # After a deliberate skip, the second watering keeps routine priority
            # until the nearest unit is running out of time.
            if unwatered >= 1 and planned_skip(CROPS[tile["crop"]], tile, world.day):
                turns_left = TURNS_PER_DAY - world.hour
                near = min((manhattan_dist(pos, u) for _, u in world.units), default=0)
                if near + WATER_DEFER_SLACK < turns_left:
                    prio = PRIO_WATER
            if is_late_wheat(tile):
                prio = PRIO_LATE_WHEAT_WATER_URGENT if unwatered >= 1 else PRIO_LATE_WHEAT_WATER
            tasks.append(TaskRequest(category="water", pos=pos, priority=prio, action=["WATER"]))

        return tasks, []

In [ ]:
%%writefile farm_agent/advisors/maintenance.py
"""Maintenance: harvest, care, collect and apply fertilizer, clear weeds."""

from typing import List, Tuple

from farm_agent.advisors.base import BaseAdvisor
from farm_agent.config import (
    CROPS,
    PRIO_APPLY_FERT_BERRY,
    PRIO_APPLY_FERT_DUE,
    PRIO_CARE,
    PRIO_COLLECT_FERT,
    PRIO_HARVEST,
    PRIO_WEED,
)
from farm_agent.messages import MarketRequest, TaskRequest
from farm_agent.world.state import WorldModel


class MaintenanceAdvisor(BaseAdvisor):

    def get_requests(self, world: WorldModel) -> Tuple[List[TaskRequest], List[MarketRequest]]:
        tasks: List[TaskRequest] = []
        tc = world.classified_tiles

        for pos in tc.harvest:
            tasks.append(TaskRequest(category="harvest", pos=pos, priority=PRIO_HARVEST,
                                     action=["HARVEST"]))
        for pos in tc.care:
            tasks.append(TaskRequest(category="care", pos=pos, priority=PRIO_CARE, action=["CARE"]))
        for pos in tc.fert:
            tasks.append(TaskRequest(category="fert", pos=pos, priority=PRIO_COLLECT_FERT,
                                     action=["COLLECT_FERTILIZER"]))
        for pos in tc.apply_fert:
            # Every planned application is due today. On an ongoing crop it pays
            # a doubled production that expires tonight.
            prio = PRIO_APPLY_FERT_BERRY if _is_ongoing(world, pos) else PRIO_APPLY_FERT_DUE
            tasks.append(TaskRequest(category="apply_fert", pos=pos, priority=prio,
                                     action=["FERTILIZE"], requires_inv={"FERTILIZER": 1}))
        for pos in tc.weed:
            tasks.append(TaskRequest(category="weed", pos=pos, priority=PRIO_WEED, action=["DIG"]))

        return tasks, []


def _is_ongoing(world, pos):
    """True for a live strawberry or tomato tile."""
    x, y = pos
    tile = world.tiles[y][x]
    if not isinstance(tile, dict) or tile.get("kind") != "PLANT":
        return False
    cd = CROPS.get(tile.get("crop"))
    return cd is not None and cd["ongoing"]

In [ ]:
%%writefile farm_agent/advisors/crop.py
"""Crop scoring and seed purchasing."""

from typing import List, Tuple

from farm_agent.advisors.base import BaseAdvisor
from farm_agent.config import CROPS, PRIO_MKT_SEED, SEED_BUFFER
from farm_agent.messages import MarketRequest
from farm_agent.world.forecast import crop_score
from farm_agent.world.pricing import wheat_self_sufficiency_tiles
from farm_agent.world.state import WorldModel


class CropAdvisor(BaseAdvisor):

    def evaluate_crops(self, world: WorldModel) -> List[Tuple[str, float]]:
        """(crop, profit per tile-day) for every crop that can still pay back, best first."""
        scored = []
        for crop in CROPS:
            score = crop_score(crop, world.day, world.farms, world.market_inv,
                               world.unlocked_shops, world.player_idx)
            if score > float("-inf"):
                scored.append((crop, score))
        scored.sort(key=lambda x: -x[1])
        return scored

    def seed_requests(self, world: WorldModel, active_crops: List[str], plant_budget: int) -> List[MarketRequest]:
        """One seed per turn per rotated crop below its buffer.

        Nothing is bought while the labour ledger forbids new plantings. Wheat's
        buffer follows the feed self-sufficiency target, since the engine voids
        every PLANT of a crop in a turn whose PLANTs exceed its seed stock.
        """
        if plant_budget <= 0:
            return []
        tc = world.classified_tiles
        wheat_buffer = max(SEED_BUFFER, wheat_self_sufficiency_tiles(tc.animal_count, len(tc.unoccupied_structure)))
        reqs: List[MarketRequest] = []
        for crop in active_crops:
            buffer = wheat_buffer if crop == "WHEAT" else SEED_BUFFER
            if world.seeds.get(crop, 0) < buffer:
                reqs.append(MarketRequest(order=["BUY_SEED", crop, 1], priority=PRIO_MKT_SEED,
                                          category="seed", cost_estimate=CROPS[crop]["seed"]))
        return reqs

In [ ]:
%%writefile farm_agent/advisors/animal.py
"""Animal scoring (for the build rotation) and livestock purchases."""

from typing import Any, Dict, List, Tuple

from farm_agent.advisors.base import BaseAdvisor
from farm_agent.messages import MarketRequest, TaskRequest
from farm_agent.world.livestock import buy_requests, planning_farms, score
from farm_agent.world.state import WorldModel


class AnimalAdvisor(BaseAdvisor):

    def evaluate_animals(self, world: WorldModel) -> Dict[str, Any]:
        """Scores of a marginal goose and of the better pasture species."""
        farms = planning_farms(world)
        cow = score(world, farms, "COW")
        sheep = score(world, farms, "SHEEP")
        goose = score(world, farms, "GOOSE")
        pasture_animal = "COW" if cow >= sheep else "SHEEP"
        return {
            "goose_score": goose,
            "pasture_animal": pasture_animal,
            "pasture_score": cow if pasture_animal == "COW" else sheep,
        }

    def get_requests(self, world: WorldModel) -> Tuple[List[TaskRequest], List[MarketRequest]]:
        return [], buy_requests(world)

In [ ]:
%%writefile farm_agent/advisors/expansion.py
"""Hiring and land purchases."""

from typing import List, Tuple

from farm_agent.advisors.base import BaseAdvisor
from farm_agent.config import (
    FARM_HAND_COST_MULT,
    HIRE_BACKLOG_PER_UNIT,
    HIRE_BATCH_MAX,
    HIRE_CASH_FLOOR,
    HIRE_DAILY_CAP,
    HIRE_LAST_HOUR,
    HIRE_SCHEDULE,
    LAND_EMPTY_THRESHOLD,
    LAND_PRICES,
    PRIO_MKT_HIRE,
    PRIO_MKT_LAND,
)
from farm_agent.messages import MarketRequest, TaskRequest
from farm_agent.world.pricing import fib
from farm_agent.world.state import WorldModel


class ExpansionAdvisor(BaseAdvisor):

    def get_requests(self, world: WorldModel) -> Tuple[List[TaskRequest], List[MarketRequest]]:
        market_orders: List[MarketRequest] = []
        tc = world.classified_tiles

        # Hiring: today's target is the schedule floor, raised by one whenever
        # the chore backlog exceeds HIRE_BACKLOG_PER_UNIT per unit (that
        # reactive part alone is capped at HIRE_DAILY_CAP).
        hires_today = world.my_farm.get("hires_today", 0)
        pending_urgent = len(tc.water) + len(tc.feed) + len(tc.care) + len(tc.fert)
        target = HIRE_SCHEDULE[min(world.day, len(HIRE_SCHEDULE) - 1)]
        if hires_today < HIRE_DAILY_CAP and pending_urgent > len(world.units) * HIRE_BACKLOG_PER_UNIT:
            target = max(target, hires_today + 1)

        # Fill the target in one batch: a hand hired at dawn costs the same as
        # one hired at noon and works a full day. Each order carries its own
        # marginal wage so the allocator can settle it on its cash ledger.
        if world.hour <= HIRE_LAST_HOUR and hires_today < target:
            budget = float(world.money)
            for k in range(min(target - hires_today, HIRE_BATCH_MAX)):
                hire_cost = FARM_HAND_COST_MULT * fib(hires_today + k)
                if budget - hire_cost < HIRE_CASH_FLOOR:
                    break
                budget -= hire_cost
                market_orders.append(MarketRequest(order=["HIRE"], priority=PRIO_MKT_HIRE,
                                                   category="hire", cost_estimate=hire_cost))

        # Land: request the next quadrant once empty tiles run out. There is
        # deliberately no cash check here. The allocator settles sells before
        # land, so a turn that starts short but sells enough can still afford
        # it; a start-of-turn check here delayed purchases by 6-7 days.
        unlocked_quadrants = world.my_farm.get("unlocked_quadrants", ["NW"])
        n_extra_land = len(unlocked_quadrants) - 1
        if n_extra_land < len(LAND_PRICES) and len(tc.empty) <= LAND_EMPTY_THRESHOLD:
            market_orders.append(MarketRequest(order=["BUY_LAND"], priority=PRIO_MKT_LAND,
                                               category="land", cost_estimate=LAND_PRICES[n_extra_land]))

        return [], market_orders

In [ ]:
%%writefile farm_agent/advisors/market.py
"""Throttled selling and the feed-wheat top-up."""

from typing import List, Tuple

from farm_agent.advisors.base import BaseAdvisor
from farm_agent.config import (
    MARKET_I0,
    MONEY_RESERVE,
    PRIO_MKT_FEED_TOPUP,
    PRIO_MKT_SELL,
    PRODUCTS,
    WHEAT_BUY_THRESHOLD_RATIO,
)
from farm_agent.messages import MarketRequest, TaskRequest
from farm_agent.world.pricing import feed_wheat_reserve, sell_qty
from farm_agent.world.state import WorldModel


class MarketAdvisor(BaseAdvisor):

    def get_requests(self, world: WorldModel) -> Tuple[List[TaskRequest], List[MarketRequest]]:
        market_orders: List[MarketRequest] = []
        tc = world.classified_tiles
        shed_total = sum(world.shed.values())

        # The shared feed reserve: every subsystem sells above it and buys up
        # to it, so no two of them trade the same wheat back and forth.
        wheat_reserve = feed_wheat_reserve(tc.animal_count, len(tc.unoccupied_structure), world.day)
        fert_use_reserve = len(tc.apply_fert)

        # Sell everything except wheat (WheatLiquidityAdvisor's job), keeping
        # the fertilizer today's applications need.
        for item in PRODUCTS:
            if item == "WHEAT":
                continue
            qty = world.shed.get(item, 0)
            if item == "FERTILIZER" and fert_use_reserve:
                qty = max(0, qty - fert_use_reserve)
            n = sell_qty(item, qty, world.market_inv.get(item, MARKET_I0), shed_total)
            if n > 0:
                market_orders.append(MarketRequest(order=["SELL", item, n], priority=PRIO_MKT_SELL,
                                                   category="sell"))

        # Feed top-up. Bought wheat can be picked up only next turn, so buy
        # early whenever an animal that will escape tonight is not covered.
        carried_wheat = sum(inv.get("WHEAT", 0) for inv in world.inventories)
        wheat_available = world.shed.get("WHEAT", 0) + carried_wheat
        urgent_unfed = any(
            isinstance(world.tiles[y][x], dict)
            and world.tiles[y][x].get("consecutive_unfed", 0) >= 1
            for x, y in tc.feed
        )
        buy_qty = 0
        if wheat_available < wheat_reserve * WHEAT_BUY_THRESHOLD_RATIO:
            buy_qty = wheat_reserve - wheat_available
        elif urgent_unfed and wheat_available < len(tc.feed) and wheat_reserve > 0:
            # A zero reserve means the season ends tonight: an escape then costs nothing.
            buy_qty = len(tc.feed) - wheat_available
        if buy_qty > 0 and world.money >= MONEY_RESERVE:
            market_orders.append(MarketRequest(
                order=["BUY_PRODUCT", "WHEAT", buy_qty], priority=PRIO_MKT_FEED_TOPUP,
                category="feed_topup", cost_estimate=buy_qty * world.prices.get("WHEAT", 0)))

        return [], market_orders

In [ ]:
%%writefile farm_agent/advisors/wheat.py
"""Wheat as a cash-flow product, not only feed.

Winning opponents sold hundreds of units of wheat a season while early
versions of this agent sold a handful: the feed reserve locked the most liquid
staple out of revenue. Wheat is on five of eight shop menus, has the gentlest
price curve, and can be bought back when short, so surplus above the shared
feed reserve is sold in steady batches.
"""

from typing import List, Tuple

from farm_agent.advisors.base import BaseAdvisor
from farm_agent.config import (
    MARKET_I0,
    MARKET_PARAMS,
    PRIO_MKT_SELL,
    SHED_HIGH_WATERMARK,
    WHEAT_SELL_MIN_PRICE_RATIO,
    WHEAT_SURPLUS_BATCH_MAX,
)
from farm_agent.messages import MarketRequest, TaskRequest
from farm_agent.world.pricing import feed_wheat_reserve, market_price
from farm_agent.world.state import WorldModel


class WheatLiquidityAdvisor(BaseAdvisor):

    def get_requests(self, world: WorldModel) -> Tuple[List[TaskRequest], List[MarketRequest]]:
        tc = world.classified_tiles
        wheat_in_shed = world.shed.get("WHEAT", 0)
        if wheat_in_shed <= 0:
            return [], []

        # Sell only above the shared reserve plus a 2-unit deadband. The top-up
        # buys *to* the reserve, so the floor must sit strictly above it or the
        # two would trade the same units back and forth.
        reserve = feed_wheat_reserve(tc.animal_count, len(tc.unoccupied_structure), world.day)
        if reserve:
            reserve += 2
        surplus = wheat_in_shed - reserve
        if surplus <= 0:
            return [], []

        # Hold rather than dump into a badly glutted market.
        price = market_price("WHEAT", world.market_inv.get("WHEAT", MARKET_I0))
        if price < WHEAT_SELL_MIN_PRICE_RATIO * MARKET_PARAMS["WHEAT"]["base"]:
            return [], []

        # The batch cap is wheat's throttle; a nearly full shed overrides it.
        batch = min(surplus, WHEAT_SURPLUS_BATCH_MAX)
        if sum(world.shed.values()) >= SHED_HIGH_WATERMARK:
            batch = surplus
        if batch <= 0:
            return [], []
        return [], [MarketRequest(order=["SELL", "WHEAT", batch], priority=PRIO_MKT_SELL, category="sell")]

In [ ]:
%%writefile farm_agent/advisors/endgame.py
"""Endgame: stop investing, harvest everything, liquidate the shed.

Only the bank balance scores; unsold goods, standing crops and animals are
worth nothing at the end. From ENDGAME_STOP_INVEST_DAY the allocator stops
discretionary buying; from ENDGAME_HARVEST_ALL_DAY every productive tile is
harvested; and the shed is sold down to the feed reserve, which tapers to zero
on the final day.
"""

from typing import List, Tuple

from farm_agent.advisors.base import BaseAdvisor
from farm_agent.config import (
    CROPS,
    ENDGAME_HARVEST_ALL_DAY,
    ENDGAME_STOP_INVEST_DAY,
    PRIO_ENDGAME_HARVEST,
    PRIO_MKT_ENDGAME_SELL,
    PRODUCTS,
)
from farm_agent.messages import MarketRequest, TaskRequest
from farm_agent.planners.late_wheat import planned_harvest_day
from farm_agent.world.pricing import feed_wheat_reserve
from farm_agent.world.state import WorldModel

PRIO_WATER_BEFORE_HARVEST = 985.0


def is_active(world: WorldModel) -> bool:
    return world.day >= ENDGAME_STOP_INVEST_DAY


class EndgameAdvisor(BaseAdvisor):

    def get_requests(self, world: WorldModel) -> Tuple[List[TaskRequest], List[MarketRequest]]:
        if not is_active(world):
            return [], []
        tasks: List[TaskRequest] = []
        market_reqs: List[MarketRequest] = []
        tc = world.classified_tiles

        if world.day >= ENDGAME_HARVEST_ALL_DAY:
            for y in range(world.board_size):
                for x in range(world.board_size):
                    tile = world.tiles[y][x]
                    if not isinstance(tile, dict):
                        continue
                    if (tile.get("kind") == "PLANT" and tile.get("yield_units", 0) > 0
                            and world.day - tile["planted_day"] >= CROPS[tile["crop"]]["first_yield_day"]):
                        cd = CROPS[tile["crop"]]
                        age = world.day - tile["planted_day"]
                        if not cd["ongoing"] and tile["yield_units"] < cd["max_yield"]:
                            # A one-time crop below its cap waits for its planned
                            # day, and is watered first while watering still adds yield.
                            if world.day < planned_harvest_day(tile):
                                continue
                            if (not tile.get("watered_today")
                                    and (cd["max_yield_day"] + 1) // 2 <= age <= cd["max_yield_day"]):
                                tasks.append(TaskRequest(category="water", pos=(x, y),
                                                         priority=PRIO_WATER_BEFORE_HARVEST,
                                                         action=["WATER"]))
                                continue
                        tasks.append(TaskRequest(category="harvest", pos=(x, y),
                                                 priority=PRIO_ENDGAME_HARVEST, action=["HARVEST"]))
                    elif (tile.get("kind") in ("COOP", "PASTURE") and tile.get("animal")
                          and tile.get("yield_units", 0) > 0 and (x, y) not in tc.deferred):
                        tasks.append(TaskRequest(category="harvest", pos=(x, y),
                                                 priority=PRIO_ENDGAME_HARVEST, action=["HARVEST"]))

        # Liquidate the shed down to the feed reserve, ahead of routine sells.
        for item in PRODUCTS:
            qty = world.shed.get(item, 0)
            if qty <= 0:
                continue
            if item == "WHEAT":
                qty = max(0, qty - feed_wheat_reserve(tc.animal_count, len(tc.unoccupied_structure), world.day))
            if qty > 0:
                market_reqs.append(MarketRequest(order=["SELL", item, qty], priority=PRIO_MKT_ENDGAME_SELL,
                                                 category="endgame_sell"))
        return tasks, market_reqs

### Allocator
Keeps one task per tile and orders the market queue by priority, because queue position *is* settlement order: sells come first and free cash and shed space for the buys behind them. Running ledgers check cash, shed space, labour and tomorrow's payroll before any purchase.

In [ ]:
%%writefile farm_agent/allocator.py
"""ResourceAllocator: the only stage allowed to say no.

Merges every advisor's proposals into (a) one approved task per tile, sorted
by priority, for the dispatcher, and (b) a market order queue checked against
running cash, shed-space and payroll ledgers and cut to the engine's limit.
"""

from typing import Any, Dict, List, Tuple

from farm_agent.advisors.animal import AnimalAdvisor
from farm_agent.advisors.crop import CropAdvisor
from farm_agent.advisors.endgame import EndgameAdvisor
from farm_agent.advisors.endgame import is_active as endgame_active
from farm_agent.advisors.expansion import ExpansionAdvisor
from farm_agent.advisors.maintenance import MaintenanceAdvisor
from farm_agent.advisors.market import MarketAdvisor
from farm_agent.advisors.survival import SurvivalAdvisor
from farm_agent.advisors.wheat import WheatLiquidityAdvisor
from farm_agent.config import (
    ANIMAL_BUILD_RESERVE,
    ANIMALS,
    CROPS,
    FARM_HAND_COST_MULT,
    HIRE_BACKLOG_PER_UNIT,
    MAX_ANIMAL_STRUCTURES,
    MAX_MARKET_ORDERS,
    MONEY_RESERVE,
    PRIO_PLACE_ANIMAL,
    PRIO_PRODUCTION,
    ROTATION_SIZE,
    SEASON_DAYS,
    SHED_CAPACITY,
    STRUCTURE_ANIMALS,
    UNIT_DAILY_TASK_CAPACITY,
)
from farm_agent.dispatcher.pathfinding import manhattan_dist
from farm_agent.messages import MarketRequest, TaskRequest
from farm_agent.planners.late_wheat import LateWheatPlanner
from farm_agent.world.pricing import fib, wheat_self_sufficiency_tiles
from farm_agent.world.state import WorldModel

# Hires considered when estimating labour capacity and tomorrow's payroll.
MAX_HIRABLE_LOOKAHEAD = 8

Option = Tuple[str, str, float]  # (kind, item, score); kind is CROP, COOP or PASTURE


class ResourceAllocator:

    def __init__(self):
        self.survival_advisor = SurvivalAdvisor()
        self.maintenance_advisor = MaintenanceAdvisor()
        self.crop_advisor = CropAdvisor()
        self.animal_advisor = AnimalAdvisor()
        self.expansion_advisor = ExpansionAdvisor()
        self.market_advisor = MarketAdvisor()
        self.wheat_advisor = WheatLiquidityAdvisor()
        self.endgame_advisor = EndgameAdvisor()
        self.late_wheat = LateWheatPlanner()

    def resolve(self, world: WorldModel) -> Tuple[List[TaskRequest], List[List[Any]]]:
        """(approved tasks, highest priority first; market orders in settlement order)."""
        alloc_options = self._strategy_rotation(world)

        tasks: List[TaskRequest] = []
        market_reqs: List[MarketRequest] = []
        for advisor in (
            self.survival_advisor,
            self.maintenance_advisor,
            self.endgame_advisor,
            self.wheat_advisor,
            self.market_advisor,
            self.animal_advisor,
            self.expansion_advisor,
        ):
            adv_tasks, adv_market = advisor.get_requests(world)
            tasks.extend(adv_tasks)
            market_reqs.extend(adv_market)

        in_endgame = endgame_active(world)
        plant_budget = 0 if in_endgame else self._plant_budget(world)
        tasks.extend(self._production_tasks(world, alloc_options, plant_budget))
        if in_endgame:
            lw_tasks, lw_market = self.late_wheat.requests(world, {t.pos for t in tasks})
            tasks.extend(lw_tasks)
            market_reqs.extend(lw_market)
        else:
            active_crops = [item for kind, item, _ in alloc_options if kind == "CROP"]
            market_reqs.extend(self.crop_advisor.seed_requests(world, active_crops, plant_budget))

        if world.day == SEASON_DAYS - 1:
            tasks = [t for t in tasks if not self._pointless_on_last_day(world, t)]
        return self._arbitrate_tasks(tasks), self._arbitrate_market(world, market_reqs)

    @staticmethod
    def _pointless_on_last_day(world: WorldModel, task: TaskRequest) -> bool:
        """No end-of-day refresh follows the final turn, so upkeep pays nothing."""
        if task.action[0] in ("FEED", "CARE", "DIG"):
            return True
        if task.action[0] not in ("WATER", "FERTILIZE"):
            return False
        tile = world.tiles[task.pos[1]][task.pos[0]]
        if not isinstance(tile, dict):
            return False
        cd = CROPS[tile["crop"]]
        return cd["ongoing"] or world.day - tile["planted_day"] < cd["first_yield_day"]

    # ------------------------------------------------------------ strategy

    def _strategy_rotation(self, world: WorldModel) -> List[Option]:
        """The top-scored production options, served round-robin over empty tiles."""
        tc = world.classified_tiles
        animal_evals = self.animal_advisor.evaluate_animals(world)
        crop_evals = self.crop_advisor.evaluate_crops(world)

        options: List[Option] = [("CROP", crop, score) for crop, score in crop_evals]
        if animal_evals["goose_score"] > float("-inf"):
            # The goose keeps its rank in the rotation, but its slot builds a
            # pasture whenever the pasture species scores higher: otherwise a
            # goose was bought merely because the free structure was a coop.
            if animal_evals["pasture_score"] >= animal_evals["goose_score"]:
                options.append(("PASTURE", animal_evals["pasture_animal"], animal_evals["goose_score"]))
            else:
                options.append(("COOP", "GOOSE", animal_evals["goose_score"]))
        if animal_evals["pasture_score"] > float("-inf"):
            options.append(("PASTURE", animal_evals["pasture_animal"], animal_evals["pasture_score"]))
        options.sort(key=lambda o: -o[2])
        alloc_options = options[:ROTATION_SIZE] if options else [("CROP", "WHEAT", 0.0)]

        # Reserve enough wheat slots to feed the herd from our own harvest.
        # Slots above one are capped by seed stock so PLANT volume never
        # outruns seeds; one slot is always kept, because seeds are only
        # restocked for crops already in the rotation.
        wheat_slots_present = sum(1 for k, i, _ in alloc_options if k == "CROP" and i == "WHEAT")
        wheat_slots_needed = wheat_self_sufficiency_tiles(tc.animal_count, len(tc.unoccupied_structure))
        wheat_target = (
            min(wheat_slots_needed, max(1, world.seeds.get("WHEAT", 0)))
            if wheat_slots_needed > 0
            else 0
        )
        if wheat_target > wheat_slots_present:
            wheat_score = next((s for c, s in crop_evals if c == "WHEAT"), 0.0)
            alloc_options = alloc_options + [
                ("CROP", "WHEAT", wheat_score)
                for _ in range(wheat_target - wheat_slots_present)
            ]
        return alloc_options

    # ---------------------------------------------------- production tasks

    @staticmethod
    def _plant_budget(world: WorldModel) -> int:
        """Labour ledger: new plantings approved while capacity covers the chores.

        Capacity counts current units plus the hands hirable right now, since
        hands vanish every evening and are re-hired on demand.
        """
        tc = world.classified_tiles
        hirable, cash = 0, world.money
        hires_today = world.my_farm.get("hires_today", 0)
        while hirable < MAX_HIRABLE_LOOKAHEAD:
            cost = FARM_HAND_COST_MULT * fib(hires_today + hirable)
            if cash - cost < MONEY_RESERVE:
                break
            cash -= cost
            hirable += 1
        capacity = (len(world.units) + hirable) * UNIT_DAILY_TASK_CAPACITY
        workload = tc.plant_count + tc.animal_count
        return max(0, capacity - workload)

    def _production_tasks(self, world: WorldModel, alloc_options: List[Option],
                          plant_budget: int) -> List[TaskRequest]:
        tasks: List[TaskRequest] = []
        tc = world.classified_tiles

        # 1. Place purchased animals, never sending two structures after one animal.
        available: Dict[str, int] = {}
        for species in ANIMALS:
            available[species] = world.shed.get(species, 0) + sum(
                inv.get(species, 0) for inv in world.inventories
            )
        for x, y, kind in tc.unoccupied_structure:
            for cand in STRUCTURE_ANIMALS[kind]:
                if available.get(cand, 0) > 0:
                    available[cand] -= 1
                    tasks.append(TaskRequest(category="place", pos=(x, y), priority=PRIO_PLACE_ANIMAL,
                                             action=["PLACE", cand], requires_inv={cand: 1}))
                    break

        # 2. Every fundable empty tile gets a PLANT or BUILD task, cycling the
        #    rotation, so units crossing the farm can plant on the way. The
        #    dispatcher enforces per-turn seed stock.
        if not alloc_options:
            return tasks
        structure_count = tc.animal_count + len(tc.unoccupied_structure)
        unit_positions = [pos for _, pos in world.units]
        empty_tiles = sorted(
            tc.empty,
            key=lambda t: min(manhattan_dist(t, p) for p in unit_positions) if unit_positions else 0,
        )
        rotation_idx = 0
        for pos in empty_tiles:
            chosen = None
            for offset in range(len(alloc_options)):
                kind, item, _score = alloc_options[(rotation_idx + offset) % len(alloc_options)]
                if kind == "CROP":
                    if plant_budget <= 0 or world.seeds.get(item, 0) <= 0:
                        continue
                    plant_budget -= 1
                    chosen = TaskRequest(category="empty", pos=pos, priority=PRIO_PRODUCTION,
                                         action=["PLANT", item], payload={"crop": item})
                else:
                    if structure_count >= MAX_ANIMAL_STRUCTURES:
                        continue
                    if world.money - ANIMALS[item]["cost"] < ANIMAL_BUILD_RESERVE:
                        continue
                    structure_count += 1
                    chosen = TaskRequest(category="empty", pos=pos, priority=PRIO_PRODUCTION,
                                         action=["BUILD_COOP"] if kind == "COOP" else ["BUILD_PASTURE"],
                                         payload={"structure": kind, "animal": item})
                rotation_idx += 1
                break
            if chosen is not None:
                tasks.append(chosen)
        return tasks

    # --------------------------------------------------------- arbitration

    @staticmethod
    def _arbitrate_tasks(tasks: List[TaskRequest]) -> List[TaskRequest]:
        """Highest priority first; one task per tile."""
        tasks.sort(key=lambda t: -t.priority)
        approved: List[TaskRequest] = []
        claimed_pos = set()
        for task in tasks:
            if task.pos in claimed_pos:
                continue
            claimed_pos.add(task.pos)
            approved.append(task)
        return approved

    def _payroll_floor(self, world: WorldModel) -> float:
        """Cash kept for tomorrow's hires before any discretionary purchase.

        Without it, a seed spree on a cash-poor morning left too little to hire
        the hands needed to water those same seeds.
        """
        tc = world.classified_tiles
        workload = len(tc.water) + len(tc.feed) + len(tc.care) + len(tc.fert)
        needed = max(0, -(-workload // HIRE_BACKLOG_PER_UNIT) - len(world.units))
        needed = min(needed, MAX_HIRABLE_LOOKAHEAD)
        wages = sum(FARM_HAND_COST_MULT * fib(h) for h in range(needed))
        return MONEY_RESERVE + wages

    def _arbitrate_market(self, world: WorldModel, reqs: List[MarketRequest]) -> List[List[Any]]:
        """Queue orders by priority under running cash and shed ledgers.

        Queue position is settlement order. Sells are credited at the current
        price before the buys behind them are checked; buys that would overflow
        the shed (the engine fails them silently) are trimmed; seeds and land
        must leave the payroll floor intact.
        """
        reqs.sort(key=lambda r: -r.priority)
        payroll_floor = self._payroll_floor(world)
        virtual_shed = sum(world.shed.values())
        cash = float(world.money)
        endgame = endgame_active(world)
        orders: List[List[Any]] = []

        for req in reqs:
            if len(orders) >= MAX_MARKET_ORDERS:
                break
            order = list(req.order)
            op = order[0]

            if op == "SELL":
                qty = int(order[2])
                virtual_shed = max(0, virtual_shed - qty)
                cash += qty * world.prices.get(order[1], 0)
                orders.append(order)

            elif op in ("BUY_PRODUCT", "BUY_ANIMAL"):
                if endgame and op == "BUY_ANIMAL":
                    continue  # cannot pay back; feed top-ups still flow
                item, qty = order[1], int(order[2])
                unit_cost = ANIMALS[item]["cost"] if op == "BUY_ANIMAL" else world.prices.get(item, 0)
                qty = min(qty, max(0, SHED_CAPACITY - virtual_shed))
                if unit_cost > 0:
                    qty = min(qty, int(cash // unit_cost))
                if qty <= 0:
                    continue
                virtual_shed += qty
                cash -= qty * unit_cost
                orders.append([op, item, qty])

            elif op == "BUY_SEED":
                if endgame and req.category != "late_seed":
                    continue
                cost = CROPS[order[1]]["seed"] * int(order[2])
                if cash - cost < payroll_floor:
                    continue
                cash -= cost
                orders.append(order)

            elif op == "HIRE":
                if endgame and world.hour > 6:
                    continue  # an afternoon hire in the endgame earns less than its wage
                if cash < req.cost_estimate:
                    continue
                cash -= req.cost_estimate
                orders.append(order)

            else:  # BUY_LAND
                if endgame:
                    continue
                if cash - req.cost_estimate < payroll_floor:
                    continue
                cash -= req.cost_estimate
                orders.append(order)

        return orders

### Dispatcher
Each turn solves one assignment problem between free units and open tasks, with cost `16 × distance − priority`, using a pure-Python Jonker–Volgenant solver. The weight was the important part: at 1, distance can never override a priority tier, and the matching earned nothing. The experiment below shows the difference.

In [ ]:
%%writefile farm_agent/dispatcher/__init__.py
"""Spatial execution: assign units to approved tasks and move them."""

In [ ]:
%%writefile farm_agent/dispatcher/pathfinding.py
"""Grid movement helpers. The farm has no obstacles, so Manhattan routing is optimal."""

from typing import Tuple


def step_toward(fx: int, fy: int, tx: int, ty: int) -> str:
    """One step toward the target: close the x gap first, then y."""
    if fx < tx:
        return "EAST"
    if fx > tx:
        return "WEST"
    if fy < ty:
        return "SOUTH"
    if fy > ty:
        return "NORTH"
    return "PASS"


def manhattan_dist(p1: Tuple[int, int], p2: Tuple[int, int]) -> int:
    return abs(p1[0] - p2[0]) + abs(p1[1] - p2[1])

In [ ]:
%%writefile farm_agent/dispatcher/assignment.py
"""Rectangular min-cost assignment (Hungarian / Jonker-Volgenant).

Pure Python on purpose: the agent runs in the Kaggle sandbox, where a missing
dependency degrades a submission silently. Matrices are small (about 16 units
by 40 tasks), so the O(n^2 m) algorithm is far below the per-turn budget.
tests/test_assignment.py checks optimality against brute force.
"""

from typing import List, Sequence

# Sentinel cost for a (unit, task) pair the unit cannot serve. Finite so the
# algorithm stays well-defined; pairs at or above it are discarded by the
# caller instead of being treated as real assignments.
UNREACHABLE = 1e9


def min_cost_assignment(cost: Sequence[Sequence[float]]) -> List[int]:
    """Returns, per row, the column it is assigned to (-1 if unassigned).

    Handles rectangular input in either orientation. Total cost of the
    returned assignment is minimal.
    """
    n = len(cost)
    if n == 0:
        return []
    m = len(cost[0])
    if m == 0:
        return [-1] * n
    if n > m:  # algorithm below needs rows <= cols; solve the transpose
        transposed = [[cost[i][j] for i in range(n)] for j in range(m)]
        col_of_row = [-1] * n
        for j, i in enumerate(_solve(transposed, m, n)):
            if i >= 0:
                col_of_row[i] = j
        return col_of_row
    return _solve([list(row) for row in cost], n, m)


def _solve(cost: List[List[float]], n: int, m: int) -> List[int]:
    """e-maxx JV shortest-augmenting-path Hungarian; requires n <= m."""
    inf = float("inf")
    u = [0.0] * (n + 1)      # row potentials
    v = [0.0] * (m + 1)      # column potentials
    p = [0] * (m + 1)        # p[j] = 1-indexed row matched to column j
    way = [0] * (m + 1)      # augmenting-path predecessor column

    for i in range(1, n + 1):
        p[0] = i
        j0 = 0
        minv = [inf] * (m + 1)
        used = [False] * (m + 1)
        while True:
            used[j0] = True
            i0 = p[j0]
            delta = inf
            j1 = 0
            row = cost[i0 - 1]
            ui = u[i0]
            for j in range(1, m + 1):
                if used[j]:
                    continue
                cur = row[j - 1] - ui - v[j]
                if cur < minv[j]:
                    minv[j] = cur
                    way[j] = j0
                if minv[j] < delta:
                    delta = minv[j]
                    j1 = j
            for j in range(m + 1):
                if used[j]:
                    u[p[j]] += delta
                    v[j] -= delta
                else:
                    minv[j] -= delta
            j0 = j1
            if p[j0] == 0:
                break
        while j0:
            j1 = way[j0]
            p[j0] = p[j1]
            j0 = j1

    result = [-1] * n
    for j in range(1, m + 1):
        if p[j]:
            result[p[j] - 1] = j - 1
    return result

In [ ]:
%%writefile farm_agent/dispatcher/dispatcher.py
"""Dispatcher: turns the approved task list into one action per unit.

Reads no strategy state and makes no economic decisions. Each turn:

  Phase 0  pinned units: the last-day walk home, units sent home by the
           liquidity planner, and deadline feeding runs
  Phase 1  a unit standing on its task tile with the required items acts now
  Phase 2  units mid-trip keep their target, so new tasks nearby cause no thrashing
  Phase 3  global min-cost assignment of free units to unclaimed tasks
  Phase 4  greedy sweep for anything the assignment could not place
"""

from typing import Any, Dict, List, Optional, Tuple

from farm_agent.config import (
    PRODUCTS,
    TERMINAL_RETURN_MIN_VALUE,
    TERMINAL_RETURN_SAFETY_TURNS,
    TERMINAL_SELL_STEP,
    TURNS_PER_DAY,
)
from farm_agent.dispatcher.assignment import UNREACHABLE, min_cost_assignment
from farm_agent.dispatcher.pathfinding import manhattan_dist, step_toward
from farm_agent.messages import TaskRequest
from farm_agent.world.state import WorldModel

PICKUP_BATCH_MAX = 4

# Exchange rate between one tile of walking and the priority scale in the
# assignment cost `DISPATCH_DISTANCE_WEIGHT * travel - priority`. Adjacent
# priority tiers are 50 apart, so at 16 about three tiles of detour outweigh
# one tier: a nearby WATER beats a distant FEED, an adjacent one never does.
#
# Tuned, not assumed. A weight that never lets distance invert a priority tier
# (1.0) earned nothing (+$758/game, t = 0.9). A sweep over 1-64 plateaued at
# 16: +$9,968/game on held-out seeds (t = 15.1, n = 480), 93% head-to-head.
# Pure distance looked best on the tuning seeds and lost on held-out ones.
# Feeding, the irreversible failure mode, became more reliable, not less.
DISPATCH_DISTANCE_WEIGHT = 16.0

# Deadline feeding: an animal unfed since yesterday escapes tonight. When the
# nearest wheat carrier has at most this many turns to spare, it is pinned to
# the feed run rather than left to the assignment.
FEED_DEADLINE_SLACK = 2
FEED_DEADLINE_LAST_DAY = 27
NO_COMMITMENTS_FROM_DAY = 24


class Dispatcher:

    def __init__(self):
        self.day: Optional[int] = None
        # unit idx -> committed task tile; hands are re-hired daily, so reset each day
        self.commitments: Dict[int, Tuple[int, int]] = {}

    def sync_memory(self, day: int, step: int):
        if step == 0 or self.day != day:
            self.commitments.clear()
            self.day = day

    def dispatch(
        self, world: WorldModel, tasks: List[TaskRequest],
        pinned: Optional[Dict[int, List[Any]]] = None,
    ) -> Tuple[List[Any], List[List[Any]]]:
        """Returns (farmer action, hand actions)."""
        self.sync_memory(world.day, world.step)

        # Phase 0 -- pinned units leave the task pool entirely.
        returning = self._terminal_returns(world)
        for idx, act in (pinned or {}).items():
            returning.setdefault(idx, act)

        rescue_tiles = set()
        if world.day <= FEED_DEADLINE_LAST_DAY:
            remaining = TURNS_PER_DAY - world.hour
            endangered = []
            for task in tasks:
                if task.action != ["FEED"]:
                    continue
                tile = world.tiles[task.pos[1]][task.pos[0]]
                if tile.get("consecutive_unfed", 0) >= 1 and not tile.get("fed_today"):
                    endangered.append(task)
            for task in endangered:
                eligible = [(manhattan_dist(pos, task.pos), idx, pos)
                            for idx, pos in world.units if idx not in returning
                            and self._inv(world, idx).get("WHEAT", 0) > 0]
                if not eligible:
                    continue
                dist, idx, pos = min(eligible)
                if not (0 <= remaining - dist - 1 <= FEED_DEADLINE_SLACK):
                    continue
                returning[idx] = (["FEED"] if dist == 0 else
                                  [step_toward(pos[0], pos[1], task.pos[0], task.pos[1])])
                rescue_tiles.add(task.pos)
        tasks = [t for t in tasks if t.action and t.pos not in rescue_tiles]
        # A plant starts one dry day behind, so one planted on the last turn of
        # the day dies at tonight's refresh. Keep the seed for tomorrow.
        if world.hour >= TURNS_PER_DAY - 1:
            tasks = [t for t in tasks if t.action[0] != "PLANT"]
        task_by_pos: Dict[Tuple[int, int], TaskRequest] = {t.pos: t for t in tasks}
        # The engine voids ALL of a crop's PLANTs in a turn that exceeds its
        # seed stock, so never issue more than the stock.
        seed_budget: Dict[str, int] = dict(world.seeds)
        # Pending demand for each carried item, for batched PICKUPs.
        item_demand: Dict[str, int] = {}
        for t in tasks:
            for item, amt in t.requires_inv.items():
                item_demand[item] = item_demand.get(item, 0) + amt

        actions: Dict[int, List[Any]] = dict(returning)
        claimed: set = set()
        free_units: Dict[int, Tuple[int, int]] = {}

        # Phase 1 -- act on the tile you are standing on.
        for idx, pos in world.units:
            if idx in returning:
                self.commitments.pop(idx, None)
                continue
            task = task_by_pos.get(pos)
            if (
                task is not None
                and pos not in claimed
                and self._missing_item(self._inv(world, idx), task) is None
                and self._consume_seed(task.action, seed_budget)
            ):
                actions[idx] = task.action
                claimed.add(pos)
                self.commitments.pop(idx, None)
            else:
                free_units[idx] = pos

        if world.day >= NO_COMMITMENTS_FROM_DAY:
            self.commitments.clear()

        # Phase 2 -- keep cross-turn commitments.
        for idx in list(free_units):
            cpos = self.commitments.get(idx)
            if cpos is None:
                continue
            task = task_by_pos.get(cpos)
            if task is None or cpos in claimed:
                self.commitments.pop(idx, None)
                continue
            act = self._move_or_fetch(world, free_units[idx], self._inv(world, idx), task,
                                      item_demand, seed_budget)
            if act is None:  # the required item is no longer obtainable
                self.commitments.pop(idx, None)
                continue
            actions[idx] = act
            claimed.add(cpos)
            free_units.pop(idx)

        # Phase 3 -- one assignment over every free unit and unclaimed task.
        # A task-by-task greedy let an early task take the one unit standing
        # next to a later task of equal priority (~1,900 such crossings a game).
        pending = [t for t in tasks if t.pos not in claimed]
        if free_units and pending:
            unit_idxs = list(free_units)
            matrix = []
            for idx in unit_idxs:
                pos = free_units[idx]
                inv = self._inv(world, idx)
                row = []
                for task in pending:
                    dist = self._effective_dist(world, pos, inv, task)
                    row.append(UNREACHABLE if dist is None
                               else DISPATCH_DISTANCE_WEIGHT * dist - task.priority)
                matrix.append(row)
            for row_i, col in enumerate(min_cost_assignment(matrix)):
                if col < 0 or matrix[row_i][col] >= UNREACHABLE:
                    continue
                idx = unit_idxs[row_i]
                task = pending[col]
                act = self._move_or_fetch(world, free_units[idx], self._inv(world, idx), task,
                                          item_demand, seed_budget)
                if act is None:
                    continue
                actions[idx] = act
                claimed.add(task.pos)
                self.commitments[idx] = task.pos
                free_units.pop(idx)

        # Phase 4 -- greedy sweep, so coverage is never lower than plain greedy.
        for task in tasks:
            if not free_units:
                break
            if task.pos in claimed:
                continue
            best: Optional[Tuple[int, int]] = None  # (distance, unit idx)
            for idx, pos in free_units.items():
                cost = self._effective_dist(world, pos, self._inv(world, idx), task)
                if cost is None:
                    continue
                if best is None or cost < best[0]:
                    best = (cost, idx)
            if best is None:
                continue
            idx = best[1]
            act = self._move_or_fetch(world, free_units[idx], self._inv(world, idx), task,
                                      item_demand, seed_budget)
            if act is None:
                continue
            actions[idx] = act
            claimed.add(task.pos)
            self.commitments[idx] = task.pos
            free_units.pop(idx)

        unit_actions = [actions.get(idx, ["PASS"]) for idx, _ in world.units]
        return unit_actions[0], unit_actions[1:]

    # ------------------------------------------------------- terminal return

    def _terminal_returns(self, world: WorldModel) -> Dict[int, List[Any]]:
        """Units walking home on the last day so their load can still be sold.

        The season stops at hour 22 of the last day and the end-of-day sweep
        that empties backpacks into the shed never runs, so goods carried at
        the end are worth nothing (+$3.7k/game once fixed). A carrier keeps
        working until the walk home only just fits before the final market
        phase, and is left alone once it can no longer arrive in time. Distance
        and slack both fall by one per turn, so a diverted unit stays diverted
        without any state of its own.
        """
        # Step from day and hour: saved replays omit `step` for the second seat.
        step = world.day * TURNS_PER_DAY + world.hour
        slack = TERMINAL_SELL_STEP - step
        if slack < 0:
            return {}

        returning: Dict[int, List[Any]] = {}
        for idx, pos in world.units:
            inv = self._inv(world, idx)
            if not inv:
                continue
            value = sum(n * world.prices.get(item, 0) for item, n in inv.items()
                        if item in PRODUCTS and n > 0)
            if value <= TERMINAL_RETURN_MIN_VALUE:
                continue
            shed_tile = world.nearest_shed_tile(pos[0], pos[1])
            dist = manhattan_dist(pos, shed_tile)
            if dist > slack or dist < slack - TERMINAL_RETURN_SAFETY_TURNS:
                continue
            returning[idx] = (["DROP"] if dist == 0
                              else [step_toward(pos[0], pos[1], shed_tile[0], shed_tile[1])])
        return returning

    # ---------------------------------------------------------------- helpers

    @staticmethod
    def _inv(world: WorldModel, idx: int) -> Dict[str, int]:
        return world.inventories[idx] if idx < len(world.inventories) else {}

    @staticmethod
    def _missing_item(inv: Dict[str, int], task: TaskRequest) -> Optional[str]:
        for item, amt in task.requires_inv.items():
            if inv.get(item, 0) < amt:
                return item
        return None

    def _effective_dist(self, world: WorldModel, pos: Tuple[int, int],
                        inv: Dict[str, int], task: TaskRequest) -> Optional[int]:
        """Travel for this unit to serve the task, via the shed if it must fetch an item."""
        missing = self._missing_item(inv, task)
        if missing is None:
            return manhattan_dist(pos, task.pos)
        if world.shed.get(missing, 0) <= 0:
            return None
        shed_tile = world.nearest_shed_tile(pos[0], pos[1])
        return manhattan_dist(pos, shed_tile) + manhattan_dist(shed_tile, task.pos)

    def _move_or_fetch(self, world: WorldModel, pos: Tuple[int, int], inv: Dict[str, int],
                       task: TaskRequest, item_demand: Dict[str, int],
                       seed_budget: Dict[str, int]) -> Optional[List[Any]]:
        """Fetch a missing item from the shed, walk to the tile, or act on it."""
        missing = self._missing_item(inv, task)
        if missing is not None:
            if world.shed.get(missing, 0) <= 0:
                return None
            if world.is_shed_adjacent(pos):
                qty = max(1, min(PICKUP_BATCH_MAX, item_demand.get(missing, 1), world.shed[missing]))
                return ["PICKUP", missing, qty]
            tx, ty = world.nearest_shed_tile(pos[0], pos[1])
            return [step_toward(pos[0], pos[1], tx, ty)]
        if pos == task.pos:
            if not self._consume_seed(task.action, seed_budget):
                return None
            return task.action
        return [step_toward(pos[0], pos[1], task.pos[0], task.pos[1])]

    @staticmethod
    def _consume_seed(action: List[Any], seed_budget: Dict[str, int]) -> bool:
        """Reserve a seed for a PLANT; True for every other action."""
        if not action or action[0] != "PLANT":
            return True
        crop = action[1]
        if seed_budget.get(crop, 0) <= 0:
            return False
        seed_budget[crop] -= 1
        return True

### Planners
Behaviours that act around the main pipeline:
- **Early cash, days 7–11:** bank milk and melons the same day when doing so funds purchases.
- **Late-season wheat:** plant wheat on idle land and labour at the end of the season.
- **Shed room:** stop the overnight shed overflow, which had been destroying about $5k of goods a game, at zero labour cost.
- **Sell order:** sell first where rival supply threatens the price most.

In [ ]:
%%writefile farm_agent/planners/__init__.py
"""Planners that act around the main pipeline: early cash, late wheat, shed room, sell order."""

In [ ]:
%%writefile farm_agent/planners/shed_room.py
"""Zero-labour fixes for the overnight shed pour.

On the last turn of each day the engine settles unit actions, then market
orders, then empties every unit's inventory into the shed up to its capacity of
100, destroying whatever does not fit. Before these fixes the pour destroyed
about $5k of goods a game; afterwards about $0.7k (report/paper.md §8.2).

Room sell     On the last turn of a pour day, sell the shed's wheat, then its
              fertilizer, up to the units the pour would otherwise destroy.
              They are the only goods the market sells back, and a buy/sell
              round trip against an unchanged market nets zero, so emptying
              them costs nothing. Market orders settle before the pour.
Defer harvest On a day whose pour is projected to overflow, leave animal
              products on their tile instead of harvesting them, provided
              tonight's production cannot be capped by doing so.

Neither fix sends a unit anywhere.
"""

from collections import Counter

from farm_agent.config import ANIMALS, MAX_MARKET_ORDERS, SHED_CAPACITY, TURNS_PER_DAY

BUYABLE = ("WHEAT", "FERTILIZER")
# The season stops after hour 22 of day 29, so its last pour is day 28's.
LAST_POUR_DAY = 28
# A product left on day d is harvested on d+1 at the earliest and sold on d+2.
LAST_DEFER_DAY = 27


def _load(inv):
    return sum(n for n in inv.values() if n > 0)


def _counter(d):
    return Counter({k: v for k, v in (d or {}).items() if v > 0})


def after_units(world, unit_actions):
    """(shed, carried) after this turn's unit actions, applied in engine order."""
    shed = _counter(world.shed)
    carried = []
    for (idx, pos), act in zip(world.units, unit_actions):
        inv = _counter(world.inventories[idx]) if idx < len(world.inventories) else Counter()
        op = act[0] if act else "PASS"
        x, y = pos
        tile = world.tiles[y][x] if 0 <= y < len(world.tiles) and 0 <= x < len(world.tiles[y]) else None
        is_animal = isinstance(tile, dict) and bool(tile.get("animal"))
        adjacent = world.is_shed_adjacent(pos)
        if op == "HARVEST" and isinstance(tile, dict) and tile.get("yield_units", 0) > 0:
            item = ANIMALS[tile["animal"]]["product"] if is_animal else tile.get("crop")
            if item:
                inv[item] += tile["yield_units"]
        elif op == "COLLECT_FERTILIZER" and is_animal and tile.get("fertilizer_available"):
            inv["FERTILIZER"] += 1
        elif op == "FEED" and is_animal and not tile.get("fed_today") and inv["WHEAT"] > 0:
            inv["WHEAT"] -= 1
        elif op == "FERTILIZE" and isinstance(tile, dict) and tile.get("kind") == "PLANT" \
                and inv["FERTILIZER"] > 0:
            inv["FERTILIZER"] -= 1
        elif op == "PICKUP" and adjacent and len(act) >= 2:
            n = min(int(act[2]) if len(act) >= 3 else 1, shed[act[1]])
            if n > 0:
                shed[act[1]] -= n
                inv[act[1]] += n
        elif op == "PLACE" and len(act) >= 2:
            item = act[1]
            if item in ANIMALS and isinstance(tile, dict) \
                    and tile.get("kind") == ANIMALS[item]["structure"] and not tile.get("animal"):
                if inv[item] > 0:
                    inv[item] -= 1
            elif adjacent:
                n = min(int(act[2]) if len(act) >= 3 else 1, inv[item],
                        max(0, SHED_CAPACITY - _load(shed)))
                if n > 0:
                    inv[item] -= n
                    shed[item] += n
        elif op == "DROP" and adjacent:
            for item, n in list(inv.items()):
                take = min(n, max(0, SHED_CAPACITY - _load(shed)))
                shed[item] += take
            inv = Counter()
        carried.append(+inv)
    return +shed, carried


def room_orders(world, unit_actions, orders):
    """Last turn of a pour day: add SELLs of buyable shed stock to cover the overflow."""
    if world.hour != TURNS_PER_DAY - 1 or world.day > LAST_POUR_DAY:
        return orders
    shed, carried = after_units(world, unit_actions)
    for order in orders:  # the market settles before the pour
        op = order[0]
        if op == "SELL" and len(order) >= 3:
            shed[order[1]] -= min(int(order[2]), shed[order[1]])
        elif op in ("BUY_PRODUCT", "BUY_ANIMAL") and len(order) >= 3:
            shed[order[1]] += min(int(order[2]), max(0, SHED_CAPACITY - _load(shed)))
    over = _load(shed) + sum(_load(inv) for inv in carried) - SHED_CAPACITY
    if over <= 0:
        return orders
    orders = [list(o) for o in orders]
    for item in BUYABLE:
        n = min(over, shed[item])
        if n <= 0:
            continue
        existing = next((o for o in orders if o[0] == "SELL" and o[1] == item), None)
        if existing is not None:
            existing[2] = int(existing[2]) + n
        elif len(orders) < MAX_MARKET_ORDERS:
            orders.append(["SELL", item, n])
        else:
            continue
        over -= n
    return orders


def _tonight(tile, day):
    """Units tonight's refresh adds to an animal tile, before the max_held cap."""
    a = ANIMALS[tile["animal"]]
    k = day + 1 - tile.get("placed_day", day) - a["first_yield_day"]
    if k < 0 or k % a["interval"]:
        return 0
    return 1 + tile.get("pending_care_bonus", 0)  # upper bound: the bonus needs feeding


def deferrable(tile, day):
    """Leaving this tile unharvested tonight loses no production and risks no escape."""
    if tile.get("consecutive_unfed", 0) >= 1:
        return False  # an escape tonight would take the held goods with it
    n = _tonight(tile, day)
    return n == 0 or tile.get("yield_units", 0) + n <= ANIMALS[tile["animal"]]["max_held"]


def projected_overflow(world, tc):
    """Units tonight's pour would destroy if every open harvest and collection is served.

    Buyable stock is sold at hour 23, so nothing in the shed is counted as kept.
    """
    carried = Counter()
    for inv in world.inventories:
        carried.update(_counter(inv))
    intake = len(tc.fert)
    for x, y in tc.harvest:
        tile = world.tiles[y][x]
        intake += tile.get("yield_units", 0) if isinstance(tile, dict) else 0
    wheat_used = min(len(tc.feed), carried["WHEAT"])
    fert_used = min(len(tc.apply_fert), carried["FERTILIZER"])
    return _load(carried) + intake - wheat_used - fert_used - SHED_CAPACITY


def deferred_harvests(world, tc):
    """Animal harvest tiles to leave standing today, largest holdings first."""
    if world.day > LAST_DEFER_DAY:
        return set()
    over = projected_overflow(world, tc)
    if over <= 0:
        return set()
    cands = []
    for x, y in tc.harvest:
        tile = world.tiles[y][x]
        if isinstance(tile, dict) and tile.get("animal") and deferrable(tile, world.day):
            cands.append((-tile.get("yield_units", 0), x, y))
    out = set()
    for neg, x, y in sorted(cands):
        if over <= 0:
            break
        out.add((x, y))
        over += neg
    return out

In [ ]:
%%writefile farm_agent/planners/sell_order.py
"""Reorder SELL slots so the sales most exposed to rival supply settle first.

The engine settles market orders in queue order. Only the order of existing
SELLs changes; quantities, purchases and their positions do not.
"""

from farm_agent.config import ANIMALS, CROPS, MARKET_I0
from farm_agent.world.pricing import market_price


def rival_supply(world, item):
    """One day of the rival's visible production of `item`, plus goods held on its tiles."""
    n = 0.0
    for seat, farm in enumerate(world.farms):
        if seat == world.player_idx:
            continue
        for row in farm.get("tiles", []):
            for tile in row:
                if not isinstance(tile, dict):
                    continue
                animal = tile.get("animal")
                crop = tile.get("crop")
                if animal:
                    a = ANIMALS[animal]
                    if item == a["product"]:
                        n += min(a["max_held"], 1 + a["interval"]) / a["interval"]
                        n += tile.get("yield_units", 0)
                    elif item == "FERTILIZER":
                        n += 0.45
                elif crop == item:
                    c = CROPS[crop]
                    cycle = c["first_yield_day"] + (c["max_yield"] - 1) * c["interval"] if c["ongoing"] else c["max_yield_day"]
                    n += c["max_yield"] / max(1, cycle)
                    n += tile.get("yield_units", 0)
    return n


def urgency(world, order):
    """(price we lose if the rival sells first, value of the sale) for a SELL order."""
    item, n = order[1], min(order[2], world.shed.get(order[1], 0))
    inv = world.market_inv.get(item, MARKET_I0)
    rival = rival_supply(world, item)
    loss = sum(market_price(item, inv + k) - market_price(item, inv + k + rival) for k in range(n))
    return loss, n * world.prices.get(item, 0)


def reorder(world, orders):
    slots = [i for i, o in enumerate(orders) if o[0] == "SELL"]
    sales = sorted((orders[i] for i in slots), key=lambda o: urgency(world, o), reverse=True)
    result = list(orders)
    for i, order in zip(slots, sales):
        result[i] = order
    return result

In [ ]:
%%writefile farm_agent/planners/liquidity.py
"""Early intraday cash: bank carried milk and melons the same day on days 7-11.

Harvests normally reach the market only after the overnight sweep into the
shed, so goods picked up on day d are sold on day d+1 and every purchase they
fund slips a day. In the cash-constrained opening that delay compounds. This
planner sends a carrier home early only when all of the following hold:

1. Funding gap. Re-running the allocator with the carried value added to cash
   approves at least MIN_GAP more spending than without it. Because it is the
   allocator's own arbitration, every cap, reserve and ledger still applies.
2. Time and price. The deposit lands by LAST_DEPOSIT_HOUR, and selling today
   instead of at tomorrow's first market loses at most MAX_PRICE_LOSS of the
   revenue to town demand still due today.
3. Labour. After the round trips, today's remaining unit-turns still cover
   TURNS_PER_TASK per pending chore.

Deposits use `PLACE <item> <n>`, which is clamped to shed room and keeps any
remainder in hand. The placed quantity is added to `world.shed` so this turn's
market plan can sell it: unit actions settle before market orders.
"""

from farm_agent.config import (
    ANIMALS,
    CROPS,
    FARM_HAND_COST_MULT,
    LAND_PRICES,
    MARKET_I0,
    SHED_CAPACITY,
    SHOPS,
    TURNS_PER_DAY,
)
from farm_agent.dispatcher.pathfinding import manhattan_dist, step_toward
from farm_agent.world.pricing import fib, market_price, sell_qty
from farm_agent.world.state import WorldModel

DAYS = (7, 8, 9, 10, 11)
ITEMS = ("MILK", "MELON")
LAST_DEPOSIT_HOUR = 18
MIN_VALUE = 300
MIN_GAP = 100
MAX_PRICE_LOSS = 0.10
TURNS_PER_TASK = 2
MAX_RETURNERS = 2


def town_demand(item, shops, start, stop):
    """Units the town removes from the market at steps start <= s < stop."""
    per_tick = sum(2 if len(SHOPS[s]) == 1 else 1 for s in shops if item in SHOPS[s])
    n = 0
    for s in range(start, stop):
        if s % 4 == 0:
            n += per_tick
        if s % TURNS_PER_DAY == 0:
            n += 1
    return n


def revenue(item, inv, n):
    """Proceeds of n units sold one at a time from inventory inv."""
    total = 0
    for _ in range(n):
        p = market_price(item, inv)
        total += p
        if p > 1:
            inv += 1
    return total


def cash_value(world, goods):
    """Sellable value of {item: n} under the market advisor's glut throttle."""
    shed_total = sum(world.shed.values())
    total = 0
    for item, n in goods.items():
        inv = world.market_inv.get(item, MARKET_I0)
        total += revenue(item, inv, sell_qty(item, n, inv, shed_total))
    return total


def merged(goods_list):
    out = {}
    for goods in goods_list:
        for item, n in goods.items():
            out[item] = out.get(item, 0) + n
    return out


def spend(world, tasks, orders):
    """Cash a resolved plan commits: its orders plus cash-gated structure builds."""
    total, hires = 0, world.my_farm.get("hires_today", 0)
    extra_land = len(world.my_farm.get("unlocked_quadrants", ["NW"])) - 1
    for order in orders:
        op = order[0]
        if op == "BUY_SEED":
            total += CROPS[order[1]]["seed"] * int(order[2])
        elif op == "BUY_ANIMAL":
            total += ANIMALS[order[1]]["cost"] * int(order[2])
        elif op == "BUY_PRODUCT":
            total += world.prices.get(order[1], 0) * int(order[2])
        elif op == "BUY_LAND" and extra_land < len(LAND_PRICES):
            total += LAND_PRICES[extra_land]
        elif op == "HIRE":
            total += FARM_HAND_COST_MULT * fib(hires)
            hires += 1
    for task in tasks:
        if task.action and task.action[0] in ("BUILD_COOP", "BUILD_PASTURE"):
            total += ANIMALS[task.payload["animal"]]["cost"]
    return total


def planned_spend(world, allocator, extra_cash):
    """What the allocator would commit this turn holding `extra_cash` more."""
    obs = dict(world.obs)
    farms = list(world.farms)
    farms[world.player_idx] = dict(world.my_farm, money=world.money + extra_cash)
    obs["farms"] = farms
    shadow = WorldModel(obs)
    tasks, orders = allocator.resolve(shadow)
    return spend(shadow, tasks, orders)


def labor_slack(world, diverted_turns):
    tc = world.classified_tiles
    pending = len(tc.water) + len(tc.feed) + len(tc.care) + len(tc.fert) + len(tc.harvest)
    turns_left = TURNS_PER_DAY - world.hour
    return len(world.units) * turns_left - diverted_turns - TURNS_PER_TASK * pending


def price_loss(world, goods, step, deposit_step):
    """Share of today's proceeds that waiting for tomorrow's first market would add."""
    now = later = 0
    next_market = (world.day + 1) * TURNS_PER_DAY
    shed_total = sum(world.shed.values())
    for item, n in goods.items():
        inv = world.market_inv.get(item, MARKET_I0)
        inv -= town_demand(item, world.unlocked_shops, step, deposit_step)
        k = sell_qty(item, n, inv, shed_total)
        now += revenue(item, inv, k)
        inv -= town_demand(item, world.unlocked_shops, deposit_step, next_market)
        later += revenue(item, inv, k)
    return (later - now) / max(1, now)


class LiquidityPlanner:
    def __init__(self):
        self.day = None
        self.committed = {}  # unit idx -> step the commitment was made

    def plan(self, world, allocator):
        """Pinned unit actions for this turn; may add this turn's deposits to world.shed."""
        if self.day != world.day:
            self.day, self.committed = world.day, {}
        if world.day not in DAYS:
            return {}
        step = world.day * TURNS_PER_DAY + world.hour
        units = dict(world.units)
        goods = {}
        for idx in units:
            inv = world.inventories[idx] if idx < len(world.inventories) else {}
            goods[idx] = {i: inv[i] for i in ITEMS if inv.get(i, 0) > 0}

        def dist(idx):
            pos = units[idx]
            return manhattan_dist(pos, world.nearest_shed_tile(pos[0], pos[1]))

        # Drop finished commitments; abandon all of them if chores would go undone.
        for idx in list(self.committed):
            if idx not in units or not goods[idx]:
                self.committed.pop(idx)
        diverted = sum(2 * dist(i) + 1 for i in self.committed)
        if self.committed and labor_slack(world, diverted) < 0:
            self.committed, diverted = {}, 0

        room = SHED_CAPACITY - sum(world.shed.values())
        base_value = cash_value(world, merged(goods[i] for i in self.committed))
        base_spend = None
        candidates = sorted((i for i in units if goods[i] and i not in self.committed),
                            key=lambda i: -cash_value(world, goods[i]) / (2 * dist(i) + 1))
        for idx in candidates:
            d = dist(idx)
            value = cash_value(world, goods[idx])
            if len(self.committed) >= MAX_RETURNERS:
                continue
            if world.hour + d > LAST_DEPOSIT_HOUR or value < MIN_VALUE or room <= 0:
                continue
            if round(price_loss(world, goods[idx], step, step + d), 4) > MAX_PRICE_LOSS:
                continue
            if labor_slack(world, diverted + 2 * d + 1) < 0:
                continue
            if base_spend is None:
                base_spend = planned_spend(world, allocator, base_value)
            gap = planned_spend(world, allocator, base_value + value) - base_spend
            if gap < MIN_GAP:
                continue
            self.committed[idx] = step
            diverted += 2 * d + 1
            base_value += value
            base_spend += gap

        actions, incoming = {}, {}
        for idx in sorted(self.committed):
            pos = units[idx]
            if world.is_shed_adjacent(pos):
                item = max(goods[idx], key=lambda i: (goods[idx][i] * world.prices.get(i, 0), i))
                n = goods[idx][item]
                take = min(n, max(0, room))
                room -= take
                actions[idx] = ["PLACE", item, n]
                if take:
                    incoming[item] = incoming.get(item, 0) + take
            else:
                tx, ty = world.nearest_shed_tile(pos[0], pos[1])
                actions[idx] = [step_toward(pos[0], pos[1], tx, ty)]
        if incoming:
            shed = dict(world.shed)
            for item, n in incoming.items():
                shed[item] = shed.get(item, 0) + n
            world.shed = shed
        return actions

In [ ]:
%%writefile farm_agent/planners/late_wheat.py
"""Late-season wheat on land and labour that would otherwise sit idle.

From ENDGAME_STOP_INVEST_DAY the allocator stops planting, but the hands it
keeps paying for go idle while wheat (on five of the eight shop menus) still
trades near its season high. On days 24-27 this planner plants wheat on empty
tiles while each extra plant is still worth LW_MIN_VALUE after its seed, priced
on the MarketBelief wheat forecast for the day the harvest can be sold, with
the price impact of plants already scheduled included.

Wheat is cheap to buy back, so its marginal value is only the spread; it pays
here precisely because the labour and land have no other use (+$2.9k/game on
a held-out panel; report/paper.md §8.2).
"""

from farm_agent.config import CROPS, PRIO_MKT_SEED, PRIO_PRODUCTION
from farm_agent.dispatcher.pathfinding import manhattan_dist
from farm_agent.messages import MarketRequest, TaskRequest
from farm_agent.world.market_belief import MarketBelief, flow_inventory
from farm_agent.world.pricing import OPTIMAL_HARVEST, market_price
from farm_agent.world.state import planned_skip

LW_FIRST_DAY = 24
LW_LAST_DAY = 27
LW_MAX_STANDING = 40
LW_MIN_VALUE = 30
LAST_HARVEST_DAY = 29


def planned_harvest_day(tile):
    """Day a one-time crop is harvested: its optimal age, or the last day."""
    return min(tile["planted_day"] + OPTIMAL_HARVEST[tile["crop"]][0], LAST_HARVEST_DAY)


def is_late_wheat(tile):
    return (isinstance(tile, dict) and tile.get("kind") == "PLANT" and tile.get("crop") == "WHEAT"
            and tile.get("planted_day", -1) >= LW_FIRST_DAY)


class LateWheatPlanner:
    def __init__(self):
        self.belief = MarketBelief()
        self.last_action = None
        self.guard = {}  # day -> True if any plant or animal was in danger at hour 0

    def observe(self, obs):
        """Per-turn update, before any decision."""
        self.belief.update(obs, self.last_action)

    def record(self, action):
        """After deciding: remember our action for next turn's trade accounting."""
        self.last_action = action

    def _guard_blocked(self, world):
        """No late wheat on a day that started with a thirsty plant or a hungry animal."""
        day = world.day
        if world.hour == 0 and day not in self.guard:
            blocked = False
            for x, y in world.classified_tiles.water:
                t = world.tiles[y][x]
                if t.get("consecutive_unwatered", 0) >= 1 and not planned_skip(CROPS[t["crop"]], t, day):
                    blocked = True
            for row in world.tiles:
                for t in row:
                    if isinstance(t, dict) and "animal" in t and t.get("consecutive_unfed", 0) >= 1:
                        blocked = True
            self.guard[day] = blocked
        return self.guard.get(day, False)

    def requests(self, world, claimed_positions):
        """(tasks, market requests) for today's late wheat; empty outside the window."""
        day = world.day
        if not (LW_FIRST_DAY <= day <= LW_LAST_DAY) or self._guard_blocked(world):
            return [], []
        tc = world.classified_tiles
        harvest_day = min(day + 4, LAST_HARVEST_DAY)
        units = harvest_day - day
        if units < 2:
            return [], []
        sale_day = min(harvest_day + 1, LAST_HARVEST_DAY)
        standing = 0
        planned_units = 0.0
        for row in world.tiles:
            for t in row:
                if is_late_wheat(t):
                    standing += 1
                    h = min(t["planted_day"] + 4, LAST_HARVEST_DAY)
                    if min(h + 1, LAST_HARVEST_DAY) <= sale_day:
                        planned_units += max(t.get("yield_units", 1), h - t["planted_day"])
        room = LW_MAX_STANDING - standing
        if room <= 0:
            return [], []
        inv = flow_inventory(self.belief, dict(world.obs), "WHEAT", max(1, sale_day - day))
        empties = [p for p in tc.empty if p not in claimed_positions]
        unit_positions = [pos for _, pos in world.units]
        empties.sort(key=lambda t: min(manhattan_dist(t, p) for p in unit_positions) if unit_positions else 0)
        n = 0
        for _ in range(min(room, len(empties))):
            value = units * market_price("WHEAT", inv + planned_units + units / 2.0) - CROPS["WHEAT"]["seed"]
            if value < LW_MIN_VALUE:
                break
            planned_units += units
            n += 1
        tasks = [TaskRequest(category="late_wheat", pos=pos, priority=PRIO_PRODUCTION,
                             action=["PLANT", "WHEAT"], payload={"crop": "WHEAT"}) for pos in empties[:n]]
        market = []
        need = n - world.seeds.get("WHEAT", 0)
        if need > 0:
            market.append(MarketRequest(order=["BUY_SEED", "WHEAT", need], priority=PRIO_MKT_SEED,
                                        category="late_seed", cost_estimate=CROPS["WHEAT"]["seed"] * need))
        return tasks, market

## Sanity checks
The agent's model of the market must match the engine it plays against, and the assignment solver must be optimal.

In [ ]:
import itertools, random
from kaggle_environments.envs.kaggriculture import kaggriculture as engine
from farm_agent import config
from farm_agent.world.pricing import market_price
from farm_agent.dispatcher.assignment import min_cost_assignment

assert config.CROPS == engine.CROPS and config.ANIMALS == engine.ANIMALS and config.SHOPS == engine.SHOPS
mismatches = sum(market_price(i, inv) != engine.market_price(i, inv)
                 for i in config.PRODUCTS for inv in range(0, 30001, 13))
print("price mismatches vs engine:", mismatches)

rng = random.Random(0)
for _ in range(200):
    n, m = rng.randint(1, 5), rng.randint(1, 5)
    cost = [[rng.randint(-99, 99) for _ in range(m)] for _ in range(n)]
    got = sum(cost[i][j] for i, j in enumerate(min_cost_assignment(cost)) if j >= 0)
    best = min(sum(cost[i][j] for i, j in (enumerate(p) if n <= m else ((r, c) for c, r in enumerate(p))))
               for p in itertools.permutations(range(max(n, m)), min(n, m)))
    assert got == best
print("assignment solver optimal on 200 random matrices")

## Play a game
`starter` is the engine's built-in baseline. Try `"random"`, or `"main.py"` for self-play.

In [ ]:
from kaggle_environments import make
import main

env = make("kaggriculture", configuration={"episodeSteps": 720, "seed": 1}, debug=True)
env.run([main.agent, "starter"])
for i, s in enumerate(env.steps[-1]):
    print(f"player {i}: ${s.reward:,.0f}  {s.status}")

In [ ]:
env.render(mode="ipython", width=1000, height=700)

## A paired experiment
The same agent plays the same 30 seeds from both seats twice: once with the dispatcher's distance weight at 1, where priorities always win, and once at 16, as shipped. Each game runs in its own process. Takes a few minutes.

On my full held-out panel (60 seeds × 4 opponents × 2 seats) the gain was **+$9,968/game**. Note the noise: one game's margin difference has a standard deviation near $27k, so with a handful of seeds this test is inconclusive either way. Change `range(30)` to `range(6)` and you will see that. Size the sample before you read the result.

In [ ]:
import json, statistics, subprocess, sys
from concurrent.futures import ThreadPoolExecutor

GAME = '''
import json, sys
sys.path.insert(0, ".")
import main
import farm_agent.dispatcher.dispatcher as dispatcher
dispatcher.DISPATCH_DISTANCE_WEIGHT = {weight}
from kaggle_environments import make
env = make("kaggriculture", configuration={{"episodeSteps": 720, "seed": {seed}}})
env.run([main.agent, "starter"] if {seat} == 0 else ["starter", main.agent])
money = [s.reward for s in env.steps[-1]]
print(json.dumps(money[{seat}] - money[1 - {seat}]))
'''

def margin(weight, seed, seat):
    out = subprocess.run([sys.executable, "-c", GAME.format(weight=weight, seed=seed, seat=seat)],
                         capture_output=True, text=True, check=True)
    return float(out.stdout.split()[-1])

pairs = [(seed, seat) for seed in range(30) for seat in (0, 1)]
with ThreadPoolExecutor(4) as pool:
    w1 = list(pool.map(lambda p: margin(1.0, *p), pairs))
    w16 = list(pool.map(lambda p: margin(16.0, *p), pairs))
delta = [b - a for a, b in zip(w1, w16)]
print(f"paired games: {len(delta)}")
print(f"mean margin  w=1: ${statistics.mean(w1):,.0f}   w=16: ${statistics.mean(w16):,.0f}")
se = statistics.stdev(delta) / len(delta) ** 0.5
print(f"paired delta: ${statistics.mean(delta):+,.0f}  95% CI [${statistics.mean(delta) - 1.96 * se:+,.0f}, "
      f"${statistics.mean(delta) + 1.96 * se:+,.0f}], w=16 better in {sum(d > 0 for d in delta)}/{len(delta)}")

## Build a submission
Kaggle expects `main.py` at the root of the archive. The file appears in this notebook's Output.

In [ ]:
!tar --exclude=__pycache__ -czf submission.tar.gz main.py farm_agent && tar -tzf submission.tar.gz | wc -l && ls -la submission.tar.gz

## What I learned

![Why early changes were unmeasurable](https://raw.githubusercontent.com/pangwangshu/competitions/main/kaggriculture/report/figures/fig4_measurement_resolution.png)

1. **Measure the ceiling of a search space before searching it.** Replaying a teacher's own labels through my interface showed a perfect clone would lose, which no amount of training could fix. Instrumenting the controller showed that a 16-million-point parameter search touched none of the 39 constants that actually gated decisions.
2. **Same seed ≠ same game.** Report how often paired games kept the same shop sequence, and pin the weed and shop randomness in your local harness if you can.
3. **Replayed opponents (tapes) flatter big changes**, because they never react. **Tie-break perturbations are not noise either**: they acted like fixed openings worth about ±$2k. Average candidate-minus-control over several tie-break salts.
4. **Compare net cash flow, not gross sales.** "They sell 3× our wheat" was mostly them reselling wheat they had bought.
5. **Size improvements against the loss distribution.** With losses this large, polishing $1–3k mechanisms could not move the rating.

![Online losses](https://raw.githubusercontent.com/pangwangshu/competitions/main/kaggriculture/report/figures/fig6_flippable_losses.png)

The strongest agents I could identify by replaying ladder seeds were whole-season plans mined from strong games: plant everything on day 1, run few hands early, get cows by day 4, build a wheat engine from day 12. My turn-by-turn allocator could not express that. If I did it again, I would add a season-plan layer on top of it.

Thanks to Kaggle and the hosts for a genuinely interesting environment, and to everyone who published notebooks: this ladder was unusually rich to learn from. The hire schedule in `config.py` borrows its shape from [Shape the Shop, Work the Pasture](https://www.kaggle.com/code/indarkarhana/shape-the-shop-work-the-pasture-top-10). Questions welcome.